In [1]:
:set -XOverloadedStrings
:set -XScopedTypeVariables
:set -XRankNTypes
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XDeriveFunctor
:set -XGeneralizedNewtypeDeriving
:load ../lib/src/Quantale.hs ../lib/src/KanExtension.hs ../lib/src/Bitopos.hs ../lib/src/Distribution.hs ../lib/src/SubjectiveModel.hs
import Quantale
import KanExtension
import Bitopos
import Distribution
import SubjectiveModel
import Data.List (sortBy, nub, minimumBy, maximumBy)
import Data.Ord (comparing, Down(..))
import Data.Maybe (fromMaybe, mapMaybe)
import qualified Data.Map.Strict as Map
import Control.Monad (filterM)
import System.IO (hSetBuffering, stdout, BufferMode(..))
hSetBuffering stdout LineBuffering
putStrLn "\x2705 SETUP OK: Subjective Modeling (Pyt'ev) + lib (Quantale/Kan/Bitopos/Distribution/SubjectiveModel)"

✅ SETUP OK: Subjective Modeling (Pyt'ev) + lib (Quantale/Kan/Bitopos/Distribution/SubjectiveModel)

# ❓ Теория Субъективного Моделирования Пытьева

> **Источник:** Ю.П. Пытьев, «Математические методы субъективного моделирования в научных исследованиях»  
> **Часть 1** — Математические и эмпирические основы, Вестник МГУ, Физика. Астрономия, 2018, №1  
> **Часть 2** — Приложения, Вестник МГУ, Физика. Астрономия, 2018, №2

---

## Содержание

| № | Раздел | Уровень |
|---|--------|---------|
| 0 | Введение: связи с Kan и Toposes | Категорное |
| 1 | Пространство субъективной модели и неопределённый элемент | Пытьев |
| 2 | Дуальные шкалы $L$ и $\bar{L}$ | Пытьев |
| 3 | Меры правдоподобия $\mathrm{Pl}$ и доверия $\mathrm{Bel}$ | Пытьев |
| 4 | Принцип относительности — группа автоморфизмов $\Gamma$ | Пытьев |
| 5 | Pl-интегралы и Bel-интегралы | Пытьев |
| 6 | Субъективная независимость НОЭ | Пытьев |
| 7 | Условные субъективные распределения | Пытьев |
| 8 | Альтернативные варианты мер Pl и Bel | Пытьев |
| 9 | Эмпирические основы: восстановление модели | Пытьев |
| 10 | Комбинирование субъективных и эмпирических данных | Пытьев |
| 11 | Энтропия субъективной неопределённости (Часть 2) | Пытьев |
| 12 | Идентификация состояний НО.НЧ.О. (Часть 2) | Пытьев |
| 13 | $[0,1]$ как Quantale: внутренняя топологическая структура | Категорное |
| 14 | Двойственность Исбелла и гипотеза эквивалентности подходов | Категорное |
| 14.5 | Изоморфность представлений — теоремы и тропы → PytevIso.ipynb | Категорное |
| 15 | Три сравнительных примера: битопос vs расширения Кана | Практика |
| 16 | Диагностика двигателя: три эксперта | Практика |
| 17 | Монада возможности — поссибилистский двойник монады Гири | Категорное |
| 18 | Обогащённая $\mathbf{X}$ и нетривиальная двойственность Исбелла | Категорное |

> **📦 Зависимости**
> **Пакеты:** `base`, `containers`
> **Библиотека курса:** `Bitopos`, `Distribution`, `KanExtension`, `Quantale`, `SubjectiveModel` (`src/lib`)
> **Расширения:** `DeriveFunctor` — deriving Functor ([→](Extensions.ipynb#deriving)) · `FlexibleContexts` — произвольные ограничения в контекстах ([→](Extensions.ipynb#flexiblecontexts)) · `FlexibleInstances` — инстансы для конкретных типов-применений ([→](Extensions.ipynb#flexibleinstances)) · `GeneralizedNewtypeDeriving` — newtype наследует инстансы обёрнутого типа ([→](Extensions.ipynb#deriving)) · `OverloadedStrings` — строковые литералы любого типа IsString ([→](Extensions.ipynb#overloadedstrings)) · `RankNTypes` — forall внутри аргументов функций ([→](Extensions.ipynb#rankntypes)) · `ScopedTypeVariables` — переменные типа из сигнатуры видны в теле ([→](Extensions.ipynb#scopedtypevariables))


## 0. Введение и зависимости

**SubjectiveModeling** — синтез двух глубоких идей из предшествующих ноутбуков:
- **[Расширения Кана](KanExtensions.ipynb)**: меры Pl и Bel суть левое (Lan) и правое (Ran) расширения Кана;
- **[Топосы](Toposes.ipynb)**: $[0,1]$ является полной алгеброй Хейтинга (фреймом, квантальным пространством),
  а пара $(\Omega_+, \Omega_-) = (\mathrm{Pl}, \mathrm{Bel})$ образует **битопос**;
- **[Неопределённость](Uncertainty.ipynb)**: тот же формальный аппарат — это **теория возможностей Пытьева**
  ($\Pi, N$ с нормировкой $\sup\tau=1$); субъективное моделирование отличается лишь интерпретацией (раздел 9).

### Таблица ключевых связей

| Понятие в этом ноутбуке | Откуда | Раздел |
|------------------------|--------|--------|
| $\mathrm{Pl}(E) = \mathrm{Lan}_J\,\tau(E)$ | [Расширения Кана](KanExtensions.ipynb) | Lan — левое расширение |
| $\mathrm{Bel}(E) = \mathrm{Ran}_J\,\bar{\tau}(E)$ | [Расширения Кана](KanExtensions.ipynb) | Ran — правое расширение |
| $[0,1]$ как фрейм (полная алгебра Хейтинга) | [Топосы](Toposes.ipynb) | T4 — внутренняя логика |
| Битопос $(\Omega_+, \Omega_-) = (\mathrm{Pl}, \mathrm{Bel})$ | [Топосы](Toposes.ipynb) | T5, T9 — битопос |
| $\tau: X \to [0,1]$ как пресноп | [Топосы](Toposes.ipynb) | T1 — предпучки |
| Двойственность Исбелла | [Расширения Кана](KanExtensions.ipynb) | Ran/Lan над $[0,1]$-Frm |
| Теория возможностей $\Pi$, $N$ ($\sup\tau=1$) | [Неопределённость](Uncertainty.ipynb) | Раздел 9 — возможность/необходимость |
| Группа автоморфизмов $\Gamma$ | — | Принцип относительности Пытьева |

### Граф зависимостей

![Граф зависимостей](../diagrams/subj/subj_deps.svg)


In [2]:
-- Концептуальная карта: SubjectiveModeling зависит от...

-- Из KanExtensions.ipynb:
-- Lan_J tau (E) = sup_{x in E} tau(x) = Pl(E)
-- Ran_J tauBar (E) = inf_{x not in E} tauBar(x) = Bel(E)

-- Из Toposes.ipynb:
-- [0,1] -- complete Heyting algebra (quantale, фрейм)
-- BiTopos: два классификатора Omega+ и Omega- <=> Pl и Bel

-- Принцип относительности Пытьева:
-- Группа Gamma = автоморфизмы [0,1] как решётки
-- <=> инвариантность под Gamma = только порядок имеет смысл

putStrLn "Карта зависимостей загружена."


Карта зависимостей загружена.

### Связь с теорией возможностей

Аппарат субъективного моделирования (правдоподобие $\mathrm{Pl}$, доверие $\mathrm{Bel}$) и
аппарат **теории возможностей** Пытьева (возможность $\Pi$, необходимость $N$) — это
**одна и та же** математическая конструкция. Общие дуальные шкалы
$L=([0,1],\max,\min)$, $\bar L=([0,1],\min,\max)$; совпадают формулы
$$\mathrm{Pl}(A)=\Pi(A)=\sup_{x\in A}\tau(x), \qquad
\mathrm{Bel}(A)=N(A)=\inf_{x\notin A}\bar\tau(x),$$
и интегралы Сугено. Возможность и необходимость вводятся **совместно** на дуальных шкалах с
самого начала; обе функции $\tau,\bar\tau$ задаются формально одинаково в обеих теориях.

Различается только **интерпретация**:

| | Теория возможностей | Субъективное моделирование |
|--|--|--|
| Природа | объективная, частотная альтернатива вероятности | выражение интуиции эксперта |
| Источник $\tau,\bar\tau$ | эмпирические частоты | суждение эксперта (эмпирика — один из вариантов) |
| Приоритет | согласие с наблюдаемыми частотами | мнение эксперта |

Базовая теория возможностей (частотное построение, построение по вероятностям, пример
$w_n=1/p^n$) — в ноутбуке [Uncertainty.ipynb](Uncertainty.ipynb), раздел 9.

# 1. Пространство Субъективной Модели и Неопределённый Элемент

> **Зачем.** Прежде чем считать, нужно договориться, *что* мы моделируем: не частоту события, а **суждение исследователя** о неслучайном объекте. Этот раздел вводит носитель таких суждений — пару мер Pl/Bel на всех подмножествах $X$.

## Определение (Пытьев 2018, Часть 1, п. 1.1)

Модельер-исследователь (МИ) задаёт **субъективную модель** случайной величины $\tilde{x}$ с множеством значений $X$ как **пространство с мерами правдоподобия и доверия**:

$$\boxed{(X,\;\mathcal{P}(X),\;\mathrm{Pl}^{\tilde{x}},\;\mathrm{Bel}^{\tilde{x}})}$$

где $\mathcal{P}(X)$ — класс всех подмножеств $X$, а меры задаются распределениями:

$$\tau^{\tilde{x}}(x) = \mathrm{Pl}^{\tilde{x}}(\tilde{x} = x), \quad x \in X, \quad \sup_{x \in X} \tau^{\tilde{x}}(x) = 1$$

$$\bar{\tau}^{\tilde{x}}(x) = \mathrm{Bel}^{\tilde{x}}(\tilde{x} = x), \quad x \in X, \quad \inf_{x \in X} \bar{\tau}^{\tilde{x}}(x) = 0$$

и для любого $E \subseteq X$:

$$\mathrm{Pl}^{\tilde{x}}(E) \;=\; \sup_{x \in E}\, \tau^{\tilde{x}}(x), \qquad \mathrm{Pl}^{\tilde{x}}(\varnothing) = 0,\quad \mathrm{Pl}^{\tilde{x}}(X) = 1$$

$$\mathrm{Bel}^{\tilde{x}}(E) \;=\; \inf_{x \in X \setminus E}\, \bar{\tau}^{\tilde{x}}(x), \qquad \mathrm{Bel}^{\tilde{x}}(X) = 1,\quad \mathrm{Bel}^{\tilde{x}}(\varnothing) = 0$$

## Интерпретация

**Неопределённый элемент (НОЭ)** $\tilde{x}$ — это модель субъективных суждений МИ о неизвестном значении $x \in X$:

- $\mathrm{Pl}^{\tilde{x}}(\tilde{x} = x)$ — насколько **относительно правдоподобно** равенство $\tilde{x} = x$
- $\mathrm{Bel}^{\tilde{x}}(\tilde{x} \neq x)$ — насколько следует **относительно доверять** неравенству $\tilde{x} \neq x$

«Относительно» означает: численные значения мер в $(0,1)$ **не допускают абсолютного содержательного толкования** — существенна лишь их упорядоченность.

## НОЭ как неопределённая высказывательная переменная (п. 1.2)

В теоретико-множественном представлении логики высказываний:
- $X$ — множество элементарных высказываний
- $\mathcal{P}(X)$ — класс всех высказываний
- Любое высказывание $a$ взаимно однозначно представлено множеством $A \subseteq X$ тех элементарных высказываний $x$, каждое из которых влечёт $a$

Правдоподобие $\mathrm{Pl}^{\tilde{x}}(A)$ — **относительное правдоподобие истинности** неопределённого высказывания $\tilde{x} \in A$.  
Доверие $\mathrm{Bel}^{\tilde{x}}(A)$ — **относительное доверие** неравенству $\tilde{x} \notin X \setminus A$ (т.е. $\tilde{x} \in A$).

## Эквивалентность нечётким мерам (Замечание 1.1)

Пространство $(X, \mathcal{P}(X), \mathrm{Pl}^{\tilde{x}}, \mathrm{Bel}^{\tilde{x}})$ **формально эквивалентно** нечёткому пространству $(X, \mathcal{P}(X), \mathrm{P}, \mathrm{N})$ с мерами **возможности** $\mathrm{P}$ и **необходимости** $\mathrm{N}$.

**Границы.** Pl/Bel — не вероятность: нет аддитивности, и $\mathrm{Pl}(E)+\mathrm{Pl}(X\setminus E)$ не обязана равняться 1. Аппарат уместен, когда частотная база отсутствует или нестабильна; при стабильном вероятностном пространстве честнее работает теория вероятностей.


# 2. Дуальные Шкалы $L$ и $\bar{L}$ — Структуры Решётки

> **Зачем.** Значения Pl и Bel живут не в «числах», а в **шкалах с порядком**; их две, и они дуальны. Без этого раздела непонятно, почему min/max, а не +/×.

## Теорема о единственности операций (Пытьев 2018, п. 1.3)

Из **принципа относительности** следует, что бинарные операции в шкалах значений мер Pl и Bel **однозначно определяются** условиями непрерывности, коммутативности и граничными условиями.

### Шкала $L$ (для правдоподобия)

$$\boxed{L = ([0,1],\, +,\, \times) = ([0,1],\, \max,\, \min)}$$

$$a + b = \max\{a, b\}, \qquad a \times b = \min\{a, b\}, \qquad a, b \in [0,1]$$

Граничные условия: $a + 0 = a \times 1 = a$, $\;a + 1 = 1$, $\;a \times 0 = 0$.

### Дуальная шкала $\bar{L}$ (для доверия)

$$\boxed{\bar{L} = ([0,1],\, \bar{+},\, \bar{\times}) = ([0,1],\, \min,\, \max)}$$

$$a\,\bar{+}\,b = \min\{a,b\}, \qquad a\,\bar{\times}\,b = \max\{a,b\}, \qquad a, b \in [0,1]$$

Граничные условия: $a\,\bar{+}\,1 = 1$, $\;a\,\bar{\times}\,0 = 0$ и пр.

### Откуда берётся дуальность?

Шкалы $L$ и $\bar{L}$ связаны **дуальным изоморфизмом** $\theta \in \Theta$, где $\Theta$ — класс строго монотонных убывающих функций $\theta: [0,1] \to [0,1]$, $\theta(0) = 1$, $\theta(1) = 0$:

$$\mathrm{Bel}^{\tilde{x}}(A) = \theta\!\left(\mathrm{Pl}^{\tilde{x}}(X \setminus A)\right), \qquad \bar{\tau}^{\tilde{x}}(x) = \theta(\tau^{\tilde{x}}(x))$$

Такое соответствие называется **дуальной согласованностью** мер Pl и Bel.

## Структура решётки

Как $L$, так и $\bar{L}$ являются **полными решётками** (complete lattices):
- В $L$: $\sup = \max$, $\inf = \min$
- В $\bar{L}$: $\sup = \min$, $\inf = \max$

### Уточнение: где настоящий билатис

Пара $(L, \bar{L})$ — это **одна** решётка с двумя взаимно обратными порядками, а не билатис: в билатисе (Гинзберг, Фиттинг) два порядка **независимы** — порядок истинности $\le_t$ и порядок информации $\le_k$. Настоящий билатис в теории Пытьева образуют **интервалы** $[\mathrm{Bel}(E), \mathrm{Pl}(E)] \subseteq [0,1]$:

$[a,b] \le_t [c,d] \iff a \le c \,\wedge\, b \le d \qquad\text{(истиннее)}$
$[a,b] \le_k [c,d] \iff a \le c \,\wedge\, d \le b \qquad\text{(информативнее: } [c,d] \subseteq [a,b])$

Выделенные элементы — в точности модели знания Пытьева (ср. п. 1.5):

| Элемент билатиса | Интервал | Интерпретация по Пытьеву |
|------------------|----------|--------------------------|
| $\top_t$ (истина) | $[1,1]$ | точное знание «событие достоверно» |
| $\bot_t$ (ложь) | $[0,0]$ | точное знание «событие невозможно» |
| $\bot_k$ (незнание) | $[0,1]$ | **абсолютное незнание**: $\mathrm{Bel}=0$, $\mathrm{Pl}=1$ |
| $\top_k$ (противоречие) | $[1,0]$ | $\mathrm{Bel} > \mathrm{Pl}$ — несогласованные данные (ср. критерий п. 2.2) |

Это структура Белнапа FOUR, растянутая на континуум — interlacing-законы проверяются в коде ниже.

**Границы.** Структуры решётки фиксируют только **порядок** — арифметика значений ($1-t$ и т.п.) есть лишь координатное представление (см. раздел 4).

In [3]:
-- | Раздел 2+: интервальный билатис — из библиотеки (../lib/Bitopos.hs)

demoBilattice :: IO ()
demoBilattice = do
  putStrLn "=== Razdel 2+: intervalnyj bilatis ==="
  putStrLn $ "  Reshyotka po <=t: " ++ show (checkLatticeLaws leqT joinT meetT)
  putStrLn $ "  Reshyotka po <=k: " ++ show (checkLatticeLaws leqK joinK meetK)
  putStrLn $ "  Interlacing:      " ++ show checkInterlacing
  putStrLn $ "  bot_k = " ++ show bUnknown ++ " (absolyutnoe neznanie)"
  putStrLn $ "  top_k = " ++ show bContra  ++ " (protivorechie, Bel > Pl)"
  putStrLn $ "  joinK neznanie tochn.znanie: " ++ show (joinK bUnknown bTrue)

demoBilattice

=== Razdel 2+: intervalnyj bilatis ===
  Reshyotka po <=t: True
  Reshyotka po <=k: True
  Interlacing:      True
  bot_k = IV {ivBel = 0.0, ivPl = 1.0} (absolyutnoe neznanie)
  top_k = IV {ivBel = 1.0, ivPl = 0.0} (protivorechie, Bel > Pl)
  joinK neznanie tochn.znanie: IV {ivBel = 1.0, ivPl = 1.0}

# 3. Меры Правдоподобия $\mathrm{Pl}$ и Доверия $\mathrm{Bel}$

> **Зачем.** Главные операции теории: как из поточечных распределений $\tau,\bar\tau$ получаются меры событий. Здесь же — формальная стыковка с теорией возможностей: тот же аппарат $\Pi/N$ из [Uncertainty.ipynb](Uncertainty.ipynb) (раздел 9), но числа выражают интуицию эксперта, а не частоту.

## Полная аддитивность мер (Пытьев 2018, п. 1.1)

Меры $\mathrm{Pl}_{\tau}$ и $\mathrm{Bel}_{\bar{\tau}}$ **вполне аддитивны** в смысле операций шкал $L$ и $\bar{L}$:

$$\mathrm{Pl}_{\tau}\!\left(\bigcup_{j \in J} E_j\right) = \sup_{j \in J} \mathrm{Pl}_{\tau}(E_j) = \bigoplus_{j \in J}^{L} \mathrm{Pl}_{\tau}(E_j)$$

$$\mathrm{Bel}_{\bar{\tau}}\!\left(\bigcap_{j \in J} E_j\right) = \inf_{j \in J} \mathrm{Bel}_{\bar{\tau}}(E_j) = \bigoplus_{j \in J}^{\bar{L}} \mathrm{Bel}_{\bar{\tau}}(E_j)$$

## Правдоподобия характеристик ОИ (п. 1.4)

Любая функция $\varphi: X \to Y$ задаёт НОЭ $\tilde{y} = \varphi(\tilde{x})$ со значениями в $Y$ и пространство $(Y, \mathcal{P}(Y), \mathrm{Pl}^{\tilde{y}}, \mathrm{Bel}^{\tilde{y}})$, где:

$$\tau^{\tilde{y}}(y) = \mathrm{Pl}^{\tilde{y}}(\tilde{y} = y) = \sup_{x: \varphi(x)=y} \tau^{\tilde{x}}(x)$$

$$\bar{\tau}^{\tilde{y}}(y) = \mathrm{Bel}^{\tilde{y}}(\tilde{y} = y) = \inf_{x: \varphi(x) \neq y} \bar{\tau}^{\tilde{x}}(x)$$

Для любого $A \subseteq Y$:

$$\mathrm{Pl}^{\tilde{y}}(A) = \sup_{y \in A} \tau^{\tilde{y}}(y), \qquad \mathrm{Bel}^{\tilde{y}}(A) = \inf_{y \notin A} \bar{\tau}^{\tilde{y}}(y)$$

> **⚠️ Тонкость двух прочтений $\bar\tau$ (доказательство и проверка — [PytevIso.ipynb](PytevIso.ipynb), §2/C7).** Формула $\bar\tau^{\tilde y}(y) = \inf_{\varphi(x)\neq y}\bar\tau(x)$ вычисляет $\mathrm{Bel}_X(\varphi^{-1}\{y\})$ — доверие к **событию** $\tilde y = y$. Но мера $\mathrm{Bel}$ восстанавливается формулой $\inf_{y\notin A}$ не из доверий к синглетонам, а из **ко-синглетонного** распределения $\bar\tau_Y(y) = \mathrm{Bel}_Y(Y{\setminus}\{y\})$ (Теорема 2 в [PytevIso.ipynb](PytevIso.ipynb): $\mathrm{Bel}$ определяется на ко-синглетонах). Подстановка «событийной» версии в $\inf_{y\notin A}$ ломается уже при $\varphi = \mathrm{id}$: для $\bar\tau = \{a{:}0.9,\ b{:}0.5,\ c{:}0.1\}$ она даёт $\mathrm{Bel}(\{b,c\}) = 0.1$ вместо $\bar\tau(a) = 0.9$. Функториально корректный образ ко-синглетонного распределения — $\bar\tau_Y(y) = \inf_{\varphi(x)=y}\bar\tau(x) = \mathrm{Ran}_\varphi\bar\tau$ (inf по слою), двойственно $\tau_Y = \mathrm{Lan}_\varphi\tau$ (sup по слою); тогда $\mathrm{Pl}_Y = \mathrm{Pl}_X\circ\varphi^{-1}$ и $\mathrm{Bel}_Y = \mathrm{Bel}_X\circ\varphi^{-1}$ — стандартные формулы образов мер возможности/необходимости (Dubois–Prade 1988). То же двойное прочтение объясняет столбец $\bar\tau$ «точного знания» в п. 1.5 ниже: как ко-синглетонное распределение ему отвечает $\bar\tau = \theta\tau = [x \neq x_0]$, а вариант $[x = x_0]$ — это синглетонные доверия. Библиотека курса (`belMeasure`) всюду использует ко-синглетонную семантику.

## Субъективные модели «абсолютного незнания» и «точного знания» (п. 1.5)

Инвариантные относительно выбора шкал $\gamma L$, $\bar{\gamma} \bar{L}$ модели:

$$\textbf{Абсолютное незнание:}\quad \tau^{\tilde{x}}(x) = 1,\; \bar{\tau}^{\tilde{x}}(x) = 0 \quad \forall x \in X$$

$$\Rightarrow \mathrm{Pl}^{\tilde{y}}(A) = 1,\; \mathrm{Bel}^{\tilde{y}}(A) = 0 \quad \forall \varphi: X \to Y,\; A \neq \varnothing$$

$$\textbf{Точное знание } x_0:\quad \tau^{\tilde{x}}(x) = \begin{cases} 1 & x = x_0 \\ 0 & x \neq x_0 \end{cases}, \quad \bar{\tau}^{\tilde{x}}(x) = \begin{cases} 1 & x = x_0 \\ 0 & x \neq x_0 \end{cases}$$

$$\Rightarrow \text{«точное знание» влечёт «точное знание» любого следствия модели}$$

**Границы.** $\sup$/$\inf$ делают меры **макситивными/минитивными**: вклад «многих средних свидетельств» не накапливается, как у суммы — это осознанная плата за работу без частот.

# 4. Принцип Относительности — Группа Автоморфизмов $\Gamma$

> **Зачем.** Если эксперт скажет «0.7», а другой «0.8» — значимо ли различие? Принцип относительности отвечает: инвариантен только **порядок**; конкретные числа — координаты.

## Группа $\Gamma$ (Пытьев 2018, п. 1.3)

**Группа автоморфизмов шкалы** $\Gamma$ — группа непрерывных строго монотонных функций:

$$\Gamma = \{\gamma: [0,1] \to [0,1] \mid \gamma(0) = 0,\; \gamma(1) = 1,\; \gamma \text{ строго монотонно возрастает}\}$$

с групповой операцией $(\gamma \circ \gamma')(a) = \gamma(\gamma'(a))$.

## Принцип относительности

Меры $\mathrm{Pl}^{\tilde{x}}$ и $\mathrm{Pl}'^{\tilde{x}}$ **эквивалентны**, если:

$$\exists\, \gamma \in \Gamma\; \forall E \in \mathcal{P}(X):\quad \gamma(\mathrm{Pl}^{\tilde{x}}(E)) = \mathrm{Pl}'^{\tilde{x}}(E)$$

Каждый МИ может формулировать модель в **своей шкале** $\gamma L$, $\bar{\gamma} \bar{L}$, $\gamma, \bar{\gamma} \in \Gamma$.

## Инвариантность автоморфизмов

Для любого $\gamma \in \Gamma$ и любых $a, b \in [0,1]$:

$$\gamma(a + b) = \gamma(a) + \gamma(b), \qquad \gamma(a \times b) = \gamma(a) \times \gamma(b)$$

$$a < b \;\Leftrightarrow\; \gamma(a) < \gamma(b)$$

Это означает: $\gamma$ — **автоморфизм решётки** $(L, \max, \min)$.

## Следствия принципа относительности

1. Численные значения $\mathrm{Pl}(E) \in (0,1)$ **не имеют абсолютного смысла** — только упорядоченность инвариантна
2. Содержательны лишь равенства $\mathrm{Pl}(E) = 0$ или $\mathrm{Pl}(E) = 1$ (не зависят от $\gamma$)
3. МИ может выбрать $\gamma$ так, чтобы значения мер были сколь угодно близки к 0 или 1

## Группы $\Gamma_S$ с выделенными значениями (п. 1.9.1)

Если коллективу МИ нужно **содержательно интерпретировать** некоторые значения $\alpha_i \in (0,1)$, они договариваются использовать подгруппу:

$$\Gamma_S = \{\gamma \in \Gamma \mid \gamma(\alpha_i) = \alpha_i,\; i = 1,\ldots,n\}$$

В частности, $\Gamma_{\{1/2\}}$ оставляет неподвижным значение $1/2$ (индифферентность МИ).

## Третий вариант: психофизическая группа (п. 1.9.2)

Группа $\Gamma'$ порождена преобразованиями $\gamma_\alpha(a) = a^\alpha$, $\alpha > 0$ (степенные функции Стивенса):

$$\gamma_\alpha(a + ' b) = \gamma_\alpha(a) +' \gamma_\alpha(b), \qquad \gamma_\alpha(a \times' b) = \gamma_\alpha(a) \times' \gamma_\alpha(b)$$

$\Gamma'$-инвариант: $\dfrac{\log \mathrm{Pl}'(A)}{\log \mathrm{Pl}'(B)} = \text{const}$ — допускает психофизическую интерпретацию.

**Границы.** Инвариантность к $\Gamma$ означает, что выводы, зависящие от арифметики значений (а не порядка), методологически нелегитимны в этой теории.


### 🔺 Категорный взгляд: Γ = Aut-группа квантали

Условия на $\gamma$ (сохранение $\max$, $\min$, порядка, концов отрезка) означают ровно одно: $\Gamma = \mathrm{Aut}\bigl([0,1], \max, \min\bigr)$ — **группа автоморфизмов квантали**. Принцип относительности тогда формулируется без слов «шкала»: субъективная модель — это объект не в $[0,1]$-значных функциях, а в **фактор-группоиде** $[\,\mathrm{Mod}/\Gamma\,]$; содержательны только $\Gamma$-инварианты (значения $0$, $1$ и порядок). Подгруппы $\Gamma_S$ с неподвижными точками $\alpha_i$ — это автоморфизмы, сохраняющие подквантали $[\alpha_i, \alpha_{i+1}]$; третий вариант мер — переход к другой квантали $([0,1], \max, \cdot)$, у которой $\mathrm{Aut}$ порождён $a \mapsto a^{\alpha}$. Действие $\Gamma$ **эквивариантно** относительно Pl: $\gamma(\mathrm{Pl}_{\tau}(E)) = \mathrm{Pl}_{\gamma\tau}(E)$ — это функториальность Lan по $\tau$, проверяется ниже.

In [4]:
-- | Раздел 4+: Gamma = Aut квантали — из библиотеки (../lib/Quantale.hs)

demoGammaAut :: IO ()
demoGammaAut = do
  putStrLn "=== Razdel 4+: Gamma kak avtomorfizmy kvantali ==="
  putStrLn $ "  t^2  - avtomorfizm:  " ++ show (isQuantaleAuto gammaSq)
  putStrLn $ "  sqrt - avtomorfizm:  " ++ show (isQuantaleAuto gammaSqrt)
  putStrLn $ "  theta - avtomorfizm: " ++ show (isQuantaleAuto theta)
             ++ " (eto dualnost, ne avtomorfizm)"
  -- эквивариантность: gamma(Pl_tau E) = Pl_{gamma.tau} E
  let dom = "abc"
      tau c = case c of { 'a' -> 1.0; 'b' -> 0.6; _ -> 0.2 }
      subs = filterM (const [True, False]) dom
      plW t e = unUI (plMeasure dom (ui . t) e)
      equiv gD = all (\e -> ui (gD (plW tau e)) =~ ui (plW (gD . tau) e)) subs
  putStrLn $ "  Ekvivariantnost Pl dlya t^2:  " ++ show (equiv (\t -> t * t))
  putStrLn $ "  Ekvivariantnost Pl dlya sqrt: " ++ show (equiv sqrt)
  let ordInv = and [ (a < b) == (gammaSq a < gammaSq b) | a <- gammaGrid, b <- gammaGrid ]
  putStrLn $ "  Poryadok invarianten: " ++ show ordInv

demoGammaAut

=== Razdel 4+: Gamma kak avtomorfizmy kvantali ===
  t^2  - avtomorfizm:  True
  sqrt - avtomorfizm:  True
  theta - avtomorfizm: False (eto dualnost, ne avtomorfizm)
  Ekvivariantnost Pl dlya t^2:  True
  Ekvivariantnost Pl dlya sqrt: True
  Poryadok invarianten: True

# 5. Pl-Интегралы и Bel-Интегралы

> **Зачем.** Чтобы принимать решения, нужны «средние» — аналоги интеграла Лебега. Здесь строятся интегралы по неаддитивным мерам (родственники интеграла Сугено).

## Определение 1.1 (Пытьев 2018, п. 1.6)

Пусть $\mathcal{L}(X)$ — класс функций $g: X \to L$ и $\bar{\mathcal{L}}(X)$ — класс функций $\bar{g}: X \to \bar{L}$.

**pl-интеграл** $\mathrm{pl}(\cdot): \mathcal{L}(X) \to L$ — **однородный и вполне аддитивный** функционал:

- **Однородность:** $\mathrm{pl}((a \times g)(\cdot)) = a \times \mathrm{pl}(g(\cdot))$ для любого $a \in [0,1]$
- **Полная аддитивность:** $\mathrm{pl}\!\left(\sum_j g_j\right)(\cdot) = \sum_j \mathrm{pl}(g_j(\cdot))$ в $L$

Аналогично **bel-интеграл** $\mathrm{bel}(\cdot): \bar{\mathcal{L}}(X) \to \bar{L}$ однороден и вполне аддитивен.

## Теорема 1.1: явное представление (Пытьев 2018, п. 1.6)

Для любого pl-интеграла $\exists!\; \tau: X \to L$ такая, что:

$$\boxed{\mathrm{pl}_{\tau}(g(\cdot)) = \sup_{x \in X} \min\{\tau(x),\, g(x)\}}$$

Для любого bel-интеграла $\exists!\; \bar{\tau}: X \to \bar{L}$ такая, что:

$$\boxed{\mathrm{bel}_{\bar{\tau}}(\bar{g}(\cdot)) = \inf_{x \in X} \max\{\bar{\tau}(x),\, \bar{g}(x)\}}$$

## Связь с мерами Pl и Bel

$$\mathrm{Pl}(E) = \mathrm{pl}_{\tau}(\chi_E(\cdot)) = \sup_{x \in E} \tau(x)$$

$$\mathrm{Bel}(E) = \mathrm{bel}_{\bar{\tau}}(\chi_E(\cdot)) = \inf_{x \in X \setminus E} \bar{\tau}(x)$$

## Интерпретация интегралов

Для функции потерь или ценности $f: X \to [0,1]$:

$$\mathrm{pl}_{\tau}(f) = \sup_{x \in X} \min\{\tau(x), f(x)\} \qquad \text{(«оптимистический» — наилучший сценарий)}$$

$$\mathrm{bel}_{\bar{\tau}}(f) = \inf_{x \in X} \max\{\bar{\tau}(x), f(x)\} \qquad \text{(«пессимистический» — наихудший сценарий)}$$

**Границы.** Pl/Bel-интегралы идемпотентны и устойчивы к выбросам, но «не чувствуют» массу повторений — там, где важна аккумуляция, нужен Лебег по вероятности.


In [5]:
-- | Разделы 1-5: субъективная модель — из библиотеки (../lib/SubjectiveModel.hs)

data State = SA | SB | SC deriving (Show, Eq, Ord, Enum, Bounded)

smDemo :: SubjModel State
smDemo = dualConsistent [SA, SB, SC] tau
  where
    tau SA = 1.0
    tau SB = 0.7
    tau SC = 0.3

demoS15 :: IO ()
demoS15 = do
  putStrLn "=== Razdely 1-5: SubjModel iz biblioteki ==="
  putStrLn $ "  Pl{SA,SB}  = " ++ show (smPl smDemo [SA, SB])
  putStrLn $ "  Bel{SA,SB} = " ++ show (smBel smDemo [SA, SB])
  putStrLn $ "  Dualno soglasovana: " ++ show (isDuallyConsistent smDemo)
  -- pl/bel-интегралы (Теорема 1.1)
  let f SA = 0.9
      f SB = 0.5
      f SC = 0.1
  putStrLn $ "  pl-integral f  = " ++ show (plIntegral [SA,SB,SC] (smTau smDemo) f)
  putStrLn $ "  bel-integral f = " ++ show (belIntegral [SA,SB,SC] (smTauBar smDemo) f)
  -- модели знания (п. 1.5)
  let ign = absoluteIgnorance [SA,SB,SC]
      knw = exactKnowledge [SA,SB,SC] SB
  putStrLn $ "  Neznanie:  Pl{SB}=" ++ show (smPl ign [SB]) ++ " Bel{SB}=" ++ show (smBel ign [SB])
  putStrLn $ "  Znanie SB: Pl{SB}=" ++ show (smPl knw [SB]) ++ " Bel{SB}=" ++ show (smBel knw [SB])
  -- образ под phi: X -> Bool
  let img = imageModel smDemo (== SA) [True, False]
  putStrLn $ "  Obraz phi: tau(True)=" ++ show (smTau img True)
             ++ " tau(False)=" ++ show (smTau img False)
  -- Полиморфизм ядра: Pl над Bool = exists, Bel = forall
  putStrLn $ "  [Bool] Pl{2,3}(x>1)  = " ++ show (plMeasure [1,2,3::Int] (> 1) [2,3])
  putStrLn $ "  [Bool] Bel{2,3}(x>1) = " ++ show (belMeasure [1,2,3::Int] (> 1) [2,3])

demoS15

=== Razdely 1-5: SubjModel iz biblioteki ===
  Pl{SA,SB}  = 1.0
  Bel{SA,SB} = 0.7
  Dualno soglasovana: True
  pl-integral f  = 0.9
  bel-integral f = 0.5
  Neznanie:  Pl{SB}=1.0 Bel{SB}=0.0
  Znanie SB: Pl{SB}=1.0 Bel{SB}=0.0
  Obraz phi: tau(True)=1.0 tau(False)=0.7
  [Bool] Pl{2,3}(x>1)  = True
  [Bool] Bel{2,3}(x>1) = False

# 6. Субъективная Независимость НОЭ

> **Зачем.** Совместные модели нескольких неопределённых элементов требуют понятия независимости — здесь оно задаётся через min/max вместо произведения.

## Определение 1.2 (Пытьев 2018, п. 1.7)

Пусть $\tilde{z} = (\tilde{z}_1, \ldots, \tilde{z}_n)$ — НОЭ со значениями в $Z_1 \times \cdots \times Z_n$.

**НОЭ $\tilde{z}_1, \ldots, \tilde{z}_n$ взаимно $\mathrm{Pl}$-независимы**, если правдоподобие события «$\forall i: \tilde{z}_i = z_i$»:

$$\tau^{\tilde{z}_1, \ldots, \tilde{z}_n}(z_1, \ldots, z_n) = \min_{1 \leq i \leq n} \tau^{\tilde{z}_i}(z_i) = \bigtimes_{i=1}^{n} \tau^{\tilde{z}_i}(z_i)$$

**НОЭ $\tilde{z}_1, \ldots, \tilde{z}_n$ взаимно $\mathrm{Bel}$-независимы**, если:

$$\bar{\tau}^{\tilde{z}_1, \ldots, \tilde{z}_n}(z_1, \ldots, z_n) = \max_{1 \leq i \leq n} \bar{\tau}^{\tilde{z}_i}(z_i) = \bar{\bigtimes}_{i=1}^{n} \bar{\tau}^{\tilde{z}_i}(z_i)$$

## Независимость событий

**События** $B_1, \ldots, B_n \in \mathcal{P}(Y)$ **взаимно $\mathrm{Pl}$-независимы**, если:

$$\mathrm{Pl}\!\left(\bigcap_{i=1}^n B_i\right) = \min_{1 \leq i \leq n} \mathrm{Pl}(B_i)$$

**$\mathrm{Bel}$-независимы**, если:

$$\mathrm{Bel}\!\left(\bigcup_{i=1}^n B_i\right) = \max_{1 \leq i \leq n} \mathrm{Bel}(B_i)$$

## Субъективная независимость (Определение 1.3)

**События $B_1, B_2$ субъективно $\mathrm{Pl}$-независимы**, если $\exists$ непрерывная $f: [0,1]^2 \to [0,1]$ такая, что:

$$\mathrm{Pl}(B_1 \cap B_2) = f(\mathrm{Pl}(B_1), \mathrm{Pl}(B_2))$$

**Теорема Пытьева:** взаимная независимость и субъективная независимость **эквивалентны**, причём $f(a,b) = \min(a,b)$.

## Связь Pl- и Bel-независимостей

Если меры Pl и Bel **дуально согласованы** ($\exists \theta \in \Theta: \mathrm{Bel}(A) = \theta(\mathrm{Pl}(X \setminus A))$), то $\mathrm{Pl}$- и $\mathrm{Bel}$-независимости **эквивалентны**.

**Границы.** Pl-независимость не влечёт Bel-независимость и наоборот; интуиция вероятностной независимости переносится лишь частично.


# 7. Условные Субъективные Распределения

> **Зачем.** Обновление суждений по наблюдению — субъективный аналог условной вероятности и основа применений (диагностика в разделе 16).

## Условное распределение правдоподобий (Пытьев 2018, п. 1.8)

Пусть $\tilde{z} = (\tilde{z}_1, \tilde{z}_2)$, $z_1 \in Z_1$, $z_2 \in Z_2$.

**Вариантом условного распределения правдоподобий** $z_1$ при условии $z_2 = z_2$ называется любое решение уравнения:

$$\min\{\tau^{\tilde{z}_1 | \tilde{z}_2}(z_1 | z_2),\; \tau^{\tilde{z}_2}(z_2)\} = \tau^{\tilde{z}_1, \tilde{z}_2}(z_1, z_2)$$

где $\tau^{\tilde{z}_2}(z_2) = \sup_{z_1 \in Z_1} \tau^{\tilde{z}_1, \tilde{z}_2}(z_1, z_2)$.

## Проблема субъективной шкалы (Определение 1.6)

Когда $0 < \tau^{\tilde{z}_2}(z_2) < 1$, условное распределение может быть не распределением правдоподобий в шкале $L$.

Решение: **субъективная шкала** $\gamma_{z_2} L$, где $\gamma_{z_2}: [0, \tau^{\tilde{z}_2}(z_2)] \to [0,1]$ — строго монотонная функция с $\gamma_{z_2}(0)=0$, $\gamma_{z_2}(\tau^{\tilde{z}_2}(z_2))=1$.

Тогда **условное правдоподобие** в субъективной шкале $\gamma_{z_2} L$:

$$\tau^{\tilde{z}_1 | \tilde{z}_2}_s(z_1 | z_2) = \gamma_{z_2} \circ \tau^{\tilde{z}_1, \tilde{z}_2}(z_1, z_2), \qquad z_1 \in Z_1$$

является распределением правдоподобий в $\gamma_{z_2} L$, в котором $z_2 = z_2$ — достоверное событие.

## Аналогия с условной вероятностью

В теории вероятностей: $\mathrm{Pr}(A | B) = \mathrm{Pr}(A \cap B) / \mathrm{Pr}(B)$ — нормирующий множитель $1/\mathrm{Pr}(B)$ задаёт «субъективную шкалу».

В теории Пытьева: переход в шкалу $\gamma_{z_2} L$ играет ту же роль, но в рамках принципа относительности.

**Границы.** Условные распределения определяются неоднозначно (есть варианты определения); выбор варианта — модельное решение.


### 🔺 Категорный взгляд: кондиционирование = residuation

Уравнение Пытьева $\min\{c,\; \tau^{\tilde{z}_2}(z_2)\} = \tau^{\tilde{z}_1,\tilde{z}_2}(z_1,z_2)$ — это вопрос о **правом сопряжённом** к функтору $\min(-, a)$ в квантали $([0,1], \min, 1)$. Его максимальное решение — в точности внутренний hom (residuation):

$$\tau^{\tilde{z}_1|\tilde{z}_2}(z_1|z_2) \;=\; \tau^{\tilde{z}_2}(z_2) \multimap \tau^{\tilde{z}_1,\tilde{z}_2}(z_1,z_2), \qquad a \multimap b = \begin{cases} 1 & a \le b \\ b & a > b \end{cases}$$

Сопряжение $\min(a,c) \le b \iff c \le (a \multimap b)$ гарантирует: (1) это решение уравнения, поскольку всегда $\tau^{\tilde{z}_1,\tilde{z}_2}(z_1,z_2) \le \tau^{\tilde{z}_2}(z_2)$; (2) оно **наибольшее** из решений; (3) $\sup_{z_1} \tau(z_1|z_2) = 1$ — условное распределение нормировано без перехода в субъективную шкалу $\gamma_{z_2}L$. Аналогия с $\Pr(A|B) = \Pr(A \cap B)/\Pr(B)$ точная: деление — это residuation квантали $([0,1], \cdot, 1)$ третьего варианта мер.

In [6]:
-- | Раздел 7+: кондиционирование = residuation (библиотека: condTau, qHom)

demoResiduation :: IO ()
demoResiduation = do
  putStrLn "=== Razdel 7+: kondicionirovanie = residuation ==="
  let dom = [(z1, z2) | z1 <- [1,2::Int], z2 <- [1,2::Int]]
      tauJ :: (Int, Int) -> Double
      tauJ (1,1) = 1.0
      tauJ (2,1) = 0.6
      tauJ (1,2) = 0.4
      tauJ (2,2) = 0.4
      tauJ _     = 0.0
      z1s = [1,2]
      z2s = [1,2]
      marg = margZ2 dom tauJ
      cnd z1 z2 = condTau dom tauJ z2 z1
  mapM_ (\(z1,z2) -> putStrLn $ "  tau(" ++ show z1 ++ "|" ++ show z2 ++ ") = "
         ++ show (cnd z1 z2)) [(a,b) | b <- z2s, a <- z1s]
  let chkEq = and [ ui (min (cnd z1 z2) (marg z2)) =~ ui (tauJ (z1, z2))
                  | z1 <- z1s, z2 <- z2s ]
      gridD = map (\k -> fromIntegral k / 20) [0..20 :: Int]
      chkMax = and [ c <= cnd z1 z2 + 1e-9
                   | z1 <- z1s, z2 <- z2s, c <- gridD
                   , ui (min c (marg z2)) =~ ui (tauJ (z1, z2)) ]
      chkNorm = and [ ui (maximum [cnd z1 z2 | z1 <- z1s]) =~ ltop | z2 <- z2s ]
  putStrLn $ "  Reshenie uravneniya: " ++ show chkEq
  putStrLn $ "  Maksimalnost:        " ++ show chkMax
  putStrLn $ "  Normirovka sup=1:    " ++ show chkNorm
  putStrLn $ "  Adjointness [0,1]:   " ++ show (checkResiduationAdj gammaGrid)
  putStrLn $ "  Adjointness Bool:    " ++ show (checkResiduationAdj [False, True])

demoResiduation

=== Razdel 7+: kondicionirovanie = residuation ===
  tau(1|1) = 1.0
  tau(2|1) = 0.6
  tau(1|2) = 1.0
  tau(2|2) = 1.0
  Reshenie uravneniya: True
  Maksimalnost:        True
  Normirovka sup=1:    True
  Adjointness [0,1]:   True
  Adjointness Bool:    True

# 8. Альтернативные Варианты Мер Pl и Bel

> **Зачем.** Первая пара (sup-min) — не единственная согласованная конструкция; здесь — систематика альтернатив и критерии выбора.

## Три варианта теории (Пытьев 2018, п. 1.9)

Пытьев рассматривает три варианта мер правдоподобия и доверия.

### Первый вариант (основной)

$$a + b = \max\{a,b\}, \quad a \times b = \min\{a,b\}$$

Группа $\Gamma$ — все непрерывные строго монотонные функции $[0,1] \to [0,1]$.  
**Числовые значения** мер, отличные от 0 и 1, **не допускают содержательной интерпретации**.

### Второй вариант (с фиксированными точками, п. 1.9.1)

Для содержательной интерпретации значений $\alpha_1, \ldots, \alpha_n \in (0,1)$ используется подгруппа $\Gamma_S \subset \Gamma$, оставляющая эти значения неподвижными.

Проекторы на интервалы $[\alpha_i, \alpha_{i+1}]$:

$$(u)^{\alpha} = \max\{\alpha, u\}, \qquad (u)_{\alpha} = \min\{\alpha, u\}$$

Шкала $L_{S'}$ разбивается на интервалы $[\alpha_i, \alpha_{i+1}]$, $i = 0, \ldots, n$, с группой $\Gamma_{S'} = \Gamma_{[\alpha_0,\alpha_1]} \otimes \cdots \otimes \Gamma_{[\alpha_n,\alpha_{n+1}]}$.

МИ могут содержательно интерпретировать **принадлежность** значений pl-интеграла неподвижным интервалам.

### Третий вариант (психофизический, п. 1.9.2)

Операции в шкале $L'$:

$$a +' b = \max\{a,b\}, \qquad a \times' b = a \cdot b \; (\text{обычное произведение})$$

Группа $\Gamma'$ порождена $\gamma_\alpha(a) = a^\alpha$, $\alpha > 0$.

Шкала $\bar{L}'$ значений $\mathrm{Bel}'$ — дуально изоморфная $L'$:

$$\theta_\beta(u) = \log_\beta u^{-1}, \quad 0 < u \leq 1, \qquad \bar{v} +' \bar{w} = \min\{\bar{v}, \bar{w}\}, \qquad \bar{v} \times' \bar{w} = \bar{v} + \bar{w}$$

Интегралы третьего варианта:

$$\mathrm{pl}'_{\tau}(f) = \max_{x \in X} (\tau(x) \cdot f(x))$$

$$\mathrm{bel}'_{\bar{\tau}}(\bar{g}) = \min_{x \in X} (\log_\beta \tau(x)^{-1} + \bar{g}(x))$$

**Инвариант третьего варианта** (допускает содержательную интерпретацию):

$$\frac{\log \mathrm{Pl}'(A)}{\log \mathrm{Pl}'(B)} = \text{const при всех } \gamma_\alpha \in \Gamma'$$

Это **отношение степеней** вероятностных оценок — аналог психофизических функций Стивенса.

**Границы.** Разные варианты дают разные численные ответы при одинаковых данных — сравнивать результаты можно только внутри одного варианта.


# 9. Эмпирические Основы: Восстановление Модели НО.НЧ.О.

> **Зачем.** Откуда брать $\tau$? Раздел показывает, как восстановить модель из упорядочивающих ответов эксперта/наблюдений — мост от слов к числам.

## Нечёткий неопределённый объект (НО.НЧ.О.)

**Неопределённый нечёткий объект** — объект, моделью которого является нечёткое пространство $(Y, \mathcal{P}(Y), \mathrm{P}(\cdot; x), \mathrm{N}(\cdot; x))$, зависящее от **неизвестного параметра** $x \in X$.

МИ предлагает субъективную модель НОЭ $\tilde{x}$ — пространство $(X, \mathcal{P}(X), \mathrm{Pl}^{\tilde{x}}, \mathrm{Bel}^{\tilde{x}})$.

## Эмпирическое оценивание по данным наблюдений (п. 2.1.2)

Если МИ доступны данные наблюдений $\eta = y_0 \in Y$ за НО.НЧ.О., то **нечёткий неопределённый элемент (НЧ.НОЭ)** $\hat{x} = \hat{x}(\eta)$, эмпирически оценивающий параметр $x \in X$, задаётся:

$$\tau^{\hat{x}}(x; y_0) = \gamma \circ g^{\eta}(y_0; x)$$

$$\bar{\tau}^{\hat{x}}(x; y_0) = \theta \circ g^{\eta}(y_0; x)$$

где $g^{\eta}(y; x)$ — распределение возможностей наблюдения $\eta = y$ при параметре $x$, $\gamma \in \Gamma$, $\theta \in \Theta$.

## Семейства оценивающих множеств максимального правдоподобия (п. 2.1.2)

$$\Phi^{-1}(y_0; \Lambda) = \{x \in X: g^{\eta}(y_0; x) \geq \min\{\Lambda, \max_{x' \in X} g^{\eta}(y_0; x')\}\}, \qquad \Lambda \in (0,1)$$

Вариант правдоподобия $\tau^{\hat{x}}(x; y_0) = \gamma \circ g^{\eta}(y_0; x)$ — **нечёткое правдоподобие** равенства $x = x \in X$ при наблюдении $\eta = y_0$.

## Правдоподобие согласия модели с данными (п. 2.3)

$$\mathrm{Pl}^{\tilde{x}}(\tilde{x} \sim \eta) = 1 - \sup\{\Lambda \in (0,1): \mathrm{Pl}^{\tilde{x}}(x \in \Phi^{-1}(\eta)) = 1\}$$

Чем меньше это значение, тем значительнее наблюдение $\eta$ **свидетельствует против** субъективной модели $(X, \mathcal{P}(X), \mathrm{Pl}^{\tilde{x}}, \mathrm{Bel}^{\tilde{x}})$.

**Границы.** Восстановление определено с точностью до $\Gamma$ (порядок!), поэтому претензии на «точные значения» здесь бессмысленны.


# 10. Комбинирование Субъективных и Эмпирических Данных

> **Зачем.** На практике есть и мнения экспертов, и данные; нужен способ их честно склеивать (используется в примере с двигателем, раздел 16).

## Задача согласованности (Пытьев 2018, п. 2.2)

Пусть $X = \{x_1, \ldots, x_m\}$, а $\tau^{(0)}(x_i)$ и $\tau^{(1)}(x_i)$ — субъективное и эмпирическое распределения правдоподобий.

Поскольку распределения задаются **в разных шкалах**, их нельзя непосредственно сравнивать числами. Можно сравнивать только **упорядоченности** значений.

## Матрица парных сравнений

Каждому распределению $\tau^{(a)}(x_i)$, $i = 1, \ldots, m$, $a = 0, 1$, сопоставляется **матрица парных сравнений** $M(a) = \{m^{(a)}_{kl}\}$:

$$m^{(a)}_{kl} = \begin{cases} 1 & \text{если } \tau^{(a)}_k > \tau^{(a)}_l \\ 0 & \text{если } \tau^{(a)}_k = \tau^{(a)}_l \\ -1 & \text{если } \tau^{(a)}_k < \tau^{(a)}_l \end{cases}$$

## Евклидово расстояние между матрицами

На классе $\mathcal{M}_{m+1}$ всех матриц парных сравнений вводится расстояние:

$$\rho(M', M'') = \left(\sum_{k,l=1}^{m+1} (m'_{kl} - m''_{kl})^2\right)^{1/2}$$

## Оптимальное совместное распределение (задача минимизации)

$$M^* = \arg\min_{M \in \mathcal{M}_{m+1}} \sum_{a=0,1} w_a^2 \rho^2(M(a), M)$$

где $w_a$ — «веса» субъективного и эмпирического распределений.

**Задача эквивалентна** нахождению матрицы парных сравнений, ближайшей к средневзвешенной матрице:

$$\bar{M} = w_0 M(0) + w_1 M(1), \qquad w_0 + w_1 = 1$$

Элементы $M^*$: $m^*_{kl} = \mathrm{sign}(\bar{m}_{kl})$ при $|\bar{m}_{kl}| \geq 1/2$, иначе $0$.

## Критерий согласованности

Если $M^* \notin \mathcal{M}_{m+1}$ (не является матрицей парных сравнений) — уровень противоречия субъективных и эмпирических данных **превышает критический**; матрице $M^*$ доверять не следует.

**Границы.** Комбинирование чувствительно к согласованности источников; противоречивые эксперты требуют отдельной обработки (взвешивание, отбраковка).


In [7]:
-- | Разделы 6-10: независимость, кондиционирование (residuation), комбинирование

sm1 :: SubjModel Int
sm1 = dualConsistent [1,2,3] (\x -> case x of { 1 -> 1.0; 2 -> 0.5; _ -> 0.3 })

sm2 :: SubjModel Bool
sm2 = dualConsistent [True, False] (\y -> if y then 1.0 else 0.4)

demoS610 :: IO ()
demoS610 = do
  putStrLn "=== Razdely 6-10: nezavisimost i kombinirovanie ==="
  putStrLn $ "  tau12(1,True)  = min(1.0,1.0) = " ++ show (plJointDist sm1 sm2 (1, True))
  putStrLn $ "  tau12(2,False) = min(0.5,0.4) = " ++ show (plJointDist sm1 sm2 (2, False))
  -- условное распределение через residuation (раздел 7+)
  let dom = [(z1, z2) | z1 <- [1,2::Int], z2 <- [1,2::Int]]
      tauJ :: (Int, Int) -> Double
      tauJ (1,1) = 1.0
      tauJ (2,1) = 0.6
      tauJ (1,2) = 0.4
      tauJ (2,2) = 0.4
      tauJ _     = 0.0
  putStrLn $ "  tau(1|2) = " ++ show (condTau dom tauJ 2 1)
  putStrLn $ "  tau(2|1) = " ++ show (condTau dom tauJ 1 2)
  -- комбинирование субъективного и эмпирического (п. 2.2)
  let subj = [1.0, 0.8, 0.5, 0.2]
      empi = [0.9, 0.7, 0.6, 0.3]
  putStrLn $ "  Rangi kombinirovaniya: " ++ show (combineDistributions subj empi 0.5 0.5)

demoS610

Line 4: Use lambda-case
Found:
\ x
  -> case x of
       1 -> 1.0
       2 -> 0.5
       _ -> 0.3
Why not:
\case
  1 -> 1.0
  2 -> 0.5
  _ -> 0.3

=== Razdely 6-10: nezavisimost i kombinirovanie ===
  tau12(1,True)  = min(1.0,1.0) = 1.0
  tau12(2,False) = min(0.5,0.4) = 0.4
  tau(1|2) = 1.0
  tau(2|1) = 0.6
  Rangi kombinirovaniya: [3.0,2.0,1.0,0.0]

# 11. Энтропия Субъективной Неопределённости (Часть 2)

> **Зачем.** Сколько неопределённости в модели? Аналог шенноновской энтропии в min/max-арифметике даёт меру информативности суждений.

## Шенноновская энтропия как отправная точка (Пытьев 2018, Часть 2, п. 2.1)

Шеннон определил энтропию как **среднюю информативность** случайного исхода испытания $(X, \mathcal{P}(X), \mathrm{Pr})$:

$$H(\mathrm{pr}_{\cdot}) = \sum_{i=1}^k \mathrm{pr}_i \log_a \frac{1}{\mathrm{pr}_i}$$

Связь с законом больших чисел (ЗБЧ): число «типичных последовательностей» длины $n$ растёт как $a^{nH}$.

## Информативность и неопределённость НОЭ (п. 2.2)

Для НОЭ $\tilde{x}$ с распределениями $\tau^{\tilde{x}}$ и $\bar{\tau}^{\tilde{x}}$ Пытьев определяет пару энтропий — формальных аналогов шенноновской:

**Информативность** НОЭ $\tilde{x}$ (в первом варианте мер Pl, Bel):

$$\boxed{H(\tau^{\tilde{x}}) = \sup_{x \in X} \min\{\tau^{\tilde{x}}(x),\; \bar{\tau}^{\tilde{x}}(x)\} = \mathrm{pl}_{\tau}(\bar{\tau}(\cdot))}$$

**Неопределённость** НОЭ $\tilde{x}$:

$$\boxed{H(\bar{\tau}^{\tilde{x}}) = \inf_{x \in X} \max\{\bar{\tau}^{\tilde{x}}(x),\; \tau^{\tilde{x}}(x)\} = \mathrm{bel}_{\bar{\tau}}(\tau(\cdot))}$$

Если меры Pl и Bel **дуально согласованы** ($\bar{\tau} = \theta \circ \tau$), то:

$$H_\theta(\tau) = \mathrm{pl}_{\tau}(\theta \circ \tau(\cdot)) = \sup_{x \in X} \min\{\tau(x),\; \theta(\tau(x))\}$$

## Свойства энтропий (формальное подобие шенноновским)

Для независимых НОЭ $\tilde{x}_1$ и $\tilde{x}_2$ (с Pl-независимым совместным распределением):

$$H_\theta(\tau^{\tilde{x}_1} \times \tau^{\tilde{x}_2}) = \max\{H_\theta(\tau^{\tilde{x}_1}),\; H_\theta(\tau^{\tilde{x}_2})\}$$

Независимо повторённое суждение **не несёт дополнительной информации** (в отличие от шенноновской, которая удваивается!). Причина: **отсутствие ЗБЧ** в первом варианте мер.

## Энтропия в третьем варианте (п. 2.3) — со своим ЗБЧ

В третьем варианте мер $\mathrm{Pl}'$ (с $\gamma_\alpha(a) = a^\alpha$):

$$H'(\tau^{\tilde{x}}) = \max_{x \in X} \bigl(\tau(x) \cdot \log_\beta \tau(x)^{-1}\bigr)$$

Здесь есть аналог ЗБЧ, и математическое ожидание субъективной информативности **связано с шенноновской энтропией**:

$$\sum_{i=1}^k \frac{\mathrm{pr}_i}{\mathrm{pr}_1} \cdot \log_a \frac{\mathrm{pr}_i}{\mathrm{pr}_1} = \delta \cdot H(\mathrm{pr}_{\cdot})$$

где $\delta = \lim_{n \to \infty} \delta(n)$ — параметр связи субъективных и вероятностных шкал.

**Границы.** Это формальный аналог: без ЗБЧ интерпретации «числа типичных последовательностей» нет; величина осмыслена как порядковая характеристика.


# 12. Идентификация Состояний НО.НЧ.О. — Оптимальное Правило (Часть 2)

> **Зачем.** Кульминация Части 2: оптимальное правило решения по субъективной модели — аналог байесовского классификатора.

## Задача идентификации (Пытьев 2018, Часть 2, Раздел 3)

**Неопределённый нечёткий объект (НО.НЧ.О.)** находится в состоянии $k \in K = \{1, \ldots, q\}$.  
Его модель $M(x) = (Z, \mathcal{P}(Z), g^{\zeta,\varkappa}(\cdot, \cdot; x))$ зависит от неизвестного параметра $x \in X$.

По наблюдению $\zeta = z \in Z$ требуется принять решение $d(z; x) \in K$ о состоянии объекта.

## Матрица «потерь» (функция возможности потерь)

Субъект, принимающий решения (СПР), задаёт возможность «потерь» $pl_{k,d} \in L$ при истинном состоянии $k$ и принятом решении $d \in K$.

## Оптимальное субъективное правило (Теорема Пытьева)

**Возможность потерь** при правиле $d(\cdot; x): Z \to K$:

$$PL(d(\cdot; x); x) = \sup_{z \in Z} \max_{k \in K} \min\{pl_{k,d(z;x)},\; g^{\zeta,\varkappa}(z, k; x)\}$$

**Оптимальное правило** $d^*(z; x)$ при каждом $x \in X$ минимизирует возможность потерь:

$$d^*(z; x) \in D^*(z; x) = \arg\min_{d \in K} \max_{k \in K} \min\{pl_{k,d},\; g^{\zeta,\varkappa}(z, k; x)\}$$

## Субъективная минимальная возможность потерь

НОЭ $\tilde{x}$ задаёт **субъективную** минимальную возможность потерь $\widetilde{pl}$ с распределением:

$$\tau^{\widetilde{pl}}(p) = \sup\{\tau^{\tilde{x}}(x) \mid x \in X,\; pl(x) = p\}$$

Качество оптимального правила $d^*(\cdot)$ характеризуется значениями:

$$p^* = \arg\max_{p \in L} \tau^{\widetilde{pl}}(p), \qquad \bar{p}^* = \arg\min_{p \in L} \bar{\tau}^{\widetilde{pl}}(p)$$

Чем меньше $p^*$ и $\bar{p}^*$, тем **лучше** оптимальное правило идентификации.

**Границы.** Оптимальность — относительно выбранных мер и функции потерь; смена варианта мер (раздел 8) меняет правило.


In [8]:
-- | Разделы 11-12: энтропии и идентификация — из библиотеки

smE :: SubjModel Int
smE = dualConsistent [1,2,3] (\x -> case x of { 1 -> 1.0; 2 -> 0.7; _ -> 0.3 })

demoS1112 :: IO ()
demoS1112 = do
  putStrLn "=== Razdely 11-12: entropii i identifikaciya ==="
  putStrLn $ "  Informativnost    = " ++ show (subjInformativity smE)
  putStrLn $ "  Neopredelennost   = " ++ show (subjUncertainty smE)
  putStrLn $ "  Dvojnaya entropiya = " ++ show (dualEntropy smE)
  putStrLn $ "  Entropiya 3-go var = " ++ show (thirdVariantEntropy smE)
  let ign = absoluteIgnorance [1,2,3::Int]
  putStrLn $ "  Neznanie: Inf=" ++ show (subjInformativity ign)
             ++ " Neopr=" ++ show (subjUncertainty ign)
  let obs k z = if k == z then 1.0 else 0.3 :: Double
      loss k d = if k == d then 0.0 else 0.8 :: Double
  putStrLn "  Optimalnoe pravilo identifikacii:"
  mapM_ (\(z, d) -> putStrLn $ "    d*(" ++ show z ++ ") = " ++ show d)
        (optimalDecision [1,2] [1,2] obs loss)

demoS1112

Line 4: Use lambda-case
Found:
\ x
  -> case x of
       1 -> 1.0
       2 -> 0.7
       _ -> 0.3
Why not:
\case
  1 -> 1.0
  2 -> 0.7
  _ -> 0.3

=== Razdely 11-12: entropii i identifikaciya ===
  Informativnost    = 0.30000000000000004
  Neopredelennost   = 0.7
  Dvojnaya entropiya = 0.30000000000000004
  Entropiya 3-go var = 0.5210896782498619
  Neznanie: Inf=0.0 Neopr=1.0
  Optimalnoe pravilo identifikacii:
    d*(1) = 1
    d*(2) = 2

# 13. $[0,1]$ как Quantale: Скрытая Топологическая Структура

> **Зачем.** Категорный взгляд: $[0,1]$ с min/⊗ — квантале, и весь аппарат Pl/Bel — обогащённая теория над ним. Это стыкует субъективное моделирование с топосной картиной ([Toposes.ipynb](Toposes.ipynb), T8: строка Sub.Mod. в таблице трёх топосов).

![Quantale structure: три проекции одного объекта](../diagrams/subj/quantale_structure.svg)

## Что такое quantale и почему $[0,1]$ им является

**Quantale** — это замкнутая моноидальная sup-решётка $(Q, \otimes, \mathbf{1})$,
в которой тензорное произведение дистрибутивно над произвольными объединениями.
Если дополнительно $\otimes = \wedge$ (meet), quantale называется **фреймом** (frame).

> **Теорема.** $([0,1], \min, 1)$ является коммутативным идемпотентным quantale (фреймом),
> а значит — **complete Heyting algebra**.

*Доказательство.* Нужно проверить дистрибутивность:
$$\min\bigl(a,\, \sup_{i} b_i\bigr) = \sup_i \min(a, b_i) \quad \forall a \in [0,1], \forall \{b_i\}$$
Это верно для любых вещественных $a, b_i \geq 0$ — стандартный факт о вещественных числах. $\square$

**Внутренний hom (residuation)** quantale $[0,1]$:
$$a \multimap b \;=\; \sup\{c \mid \min(a,c) \leq b\} \;=\; \begin{cases} 1 & a \leq b \\ b & a > b \end{cases}$$

Это именно импликация Хейтинга $a \Rightarrow b$! Таким образом, quantale $[0,1]$ несёт в себе
интуиционистскую логику — и это не случайно.

## Две топологии Скотта: откуда берётся битопос

Каждый частично упорядоченный тип (poset) порождает **топологию Скотта**: открытые множества —
верхние множества (upset), замкнутые относительно супремумов направленных семейств.
На $[0,1]$ есть **два естественных порядка**, каждый даёт свою топологию:

| Порядок | Топология Скотта | Открытые базисные множества | Интерпретация |
|---------|-----------------|----------------------------|---------------|
| $\leq$ (стандартный) | $\mathcal{T}_{\uparrow}$ | $(t, 1]$ для $t \in [0,1]$ | «высокое правдоподобие» |
| $\geq$ (обратный) | $\mathcal{T}_{\downarrow}$ | $[0, t)$ для $t \in [0,1]$ | «низкое доверие» |

Таким образом $([0,1], \mathcal{T}_{\uparrow}, \mathcal{T}_{\downarrow})$ — битопологическое пространство.

## Как $\tau$ индуцирует битопос на $X$

![Топологии Скотта: tau индуцирует битопос](../diagrams/subj/scott_topologies.svg)

Функция $\tau: X \to [0,1]$ **функториально** порождает через прообразы две топологии на $X$:

$$\mathcal{T}_{\uparrow}^X = \{\tau^{-1}(U) \mid U \in \mathcal{T}_{\uparrow}\} = \bigl\{\{x : \tau(x) > t\} \mid t \in [0,1]\bigr\}$$

$$\mathcal{T}_{\downarrow}^X = \{\tau^{-1}(U) \mid U \in \mathcal{T}_{\downarrow}\} = \bigl\{\{x : \tau(x) < t\} \mid t \in [0,1]\bigr\}$$

**Конкретный пример:** $X = \{a,b,c\}$, $\tau(a)=1.0$, $\tau(b)=0.6$, $\tau(c)=0.3$.

Открытые в $\mathcal{T}_{\uparrow}^X$: $\emptyset$, $\{a\}$ (при $t=0.6$), $\{a,b\}$ (при $t=0.3$), $\{a,b,c\}$.

Открытые в $\mathcal{T}_{\downarrow}^X$: $\emptyset$, $\{c\}$ (при $t=0.3$), $\{b,c\}$ (при $t=0.6$), $\{a,b,c\}$.

## Теорема: Pl = Interior, Bel = Closure

> **Теорема.** Для дискретного $X$ и $E \subseteq X$:
> $$\mathrm{Pl}(E) = \sup_{x \in E} \tau(x) = \mathrm{Int}_{\mathcal{T}_{\uparrow}^X}(E)$$
> $$\mathrm{Bel}(E) = \inf_{x \notin E} \bar{\tau}(x) = \mathrm{Cl}_{\mathcal{T}_{\downarrow}^X}(E)$$

*Доказательство для Pl:*
$\mathrm{Int}_{\mathcal{T}_{\uparrow}}(E) = \sup\{t : (t,1] \cap X \subseteq E\}$.
Наибольшее такое $t$ — это $\sup_{x \notin E} \tau(x)$... нет, это
$\sup_{x \in E} \tau(x)$ потому что $(t,1] \cap X \subseteq E$ тогда и только тогда когда
все $x$ с $\tau(x) > t$ лежат в $E$, то есть $t \geq \sup_{x \notin E} \tau(x)$.
В итоге $\mathrm{Int}(E) = \sup_{x \in E} \tau(x)$. $\square$

**Пример:** $E = \{a,b\}$, $\tau(a)=1.0$, $\tau(b)=0.6$, $\tau(c)=0.3$:
$\mathrm{Pl}(\{a,b\}) = \sup(1.0, 0.6) = 1.0$, $\quad \mathrm{Bel}(\{a,b\}) = \inf(\bar{\tau}(c)) = \inf(0.7) = 0.7$.

## Связь с алгеброй Хейтинга: от quantale к логике

Поскольку $[0,1]$ — complete Heyting algebra, **каждая из двух топологий** несёт
свою Heyting/co-Heyting структуру:

$$\mathcal{T}_{\uparrow}^X \text{ (Heyting):} \quad a \Rightarrow b = \begin{cases} 1 & a \leq b \\ b & a > b \end{cases}$$

$$\mathcal{T}_{\downarrow}^X \text{ (co-Heyting/Brouwer):} \quad a \leftarrow b = \begin{cases} 0 & a \geq b \\ b & a < b \end{cases}$$

Вместе $(\mathcal{T}_{\uparrow}^X, \mathcal{T}_{\downarrow}^X)$ образуют **BiHeyting-алгебру** на $[0,1]$.
Это структура, специфичная именно для *битопологических* пространств — и она возникает здесь
не ad hoc, а как следствие того, что quantale $[0,1]$ сам является self-dual объектом
(его порядок изоморфен обратному через отображение $x \mapsto 1-x$).

**Границы.** Обогащение над $[0,1]$ — не элементарный топос (нет классификатора в строгом смысле); это родственная, но другая структура (см. BiTopos в T5/T9).


# 14. Двойственность Исбелла и Гипотеза Единства Подходов

> **Зачем.** Гипотеза единства: Pl/Bel как левое/правое расширения Кана и двойственность Исбелла — общая рамка, связывающая все подходы курса ([KanExtensions.ipynb](KanExtensions.ipynb)).

## Isbell duality: пресноп ↔ копресноп

![Isbell duality над [0,1]-Frm](../diagrams/subj/isbell_duality.svg)

Для малой $\mathcal{V}$-обогащённой категории $\mathcal{C}$ существует каноническое сопряжение
(adjunction) между пресноп и копресноп — **двойственность Исбелла** (Isbell duality, 1966):

$$\bigl(\mathcal{O} \dashv \mathrm{Spec}\bigr)\;:\;
[\mathcal{C},\,\mathcal{V}]^{\mathrm{op}}
\underset{\mathcal{O}}{\overset{\mathrm{Spec}}{\rightleftharpoons}}
[\mathcal{C}^{\mathrm{op}},\,\mathcal{V}]$$

где $\mathcal{O}(X)(c) = [\mathcal{C}^{\mathrm{op}}, \mathcal{V}]\bigl(X,\, \mathcal{C}(-,c)\bigr)$ и
$\mathrm{Spec}(A)(c) = [\mathcal{C}, \mathcal{V}]\bigl(A,\, \mathcal{C}(c,-)\bigr)$.

**Ключевой факт из nLab:** $\mathrm{Spec}$ — это *левое расширение Кана вдоль вложения Йонеды*,
$\mathcal{O}$ — *левое расширение контравариантного вложения Йонеды*.
Двойственность Исбелла — общий шаблон для дуальностей Гельфанда, Стоуна, теоремы Серра-Свана.

## Применение к теории Пытьева: три уровня

Выберем $\mathcal{V} = ([0,1], \min, 1)$ — quantale/фрейм. Три уровня конкретизации:

**Уровень 1 — объекты:**
- $\mathbf{X}$ — дискретная $[0,1]$-обогащённая категория: $\mathbf{X}(x,y) = [x=y]$
- $\tau: \mathbf{X} \to \mathbf{1}$ — $[0,1]$-обогащённый функтор, задающий **пресноп правдоподобия**
- $\bar{\tau}: \mathbf{X} \to \mathbf{1}$ — двойственный **копресноп доверия**
- $J: \mathbf{X} \hookrightarrow \mathcal{P}(X)$ — включение точек в категорию подмножеств

**Уровень 2 — расширения Кана как coend/end:**

Формула левого расширения Кана через coend:
$$\mathrm{Lan}_J\,\tau\,(E) = \int^{x \in \mathbf{X}} \mathcal{P}(X)(J(x), E) \otimes_{\min} \tau(x)$$

При дискретном $X$: $\mathcal{P}(X)(J(x), E) = [x \in E] \in \{0,1\}$, тензор $\otimes = \min$, coend = $\sup$:

$$\mathrm{Lan}_J\,\tau\,(E) = \sup_{x \in E} \min(1, \tau(x)) = \sup_{x \in E} \tau(x) = \mathrm{Pl}(E)$$

Аналогично, **правое расширение Кана** через end:
$$\mathrm{Ran}_J\,\bar{\tau}\,(E) = \int_{x \in \mathbf{X}} [0,1]\bigl(\mathcal{P}(X)(E, J(x)),\, \bar{\tau}(x)\bigr)$$

**Где была дыра.** Если брать Ran вдоль того же профунктора $[x \in E]$, end даёт $\inf_{x \in E} \bar{\tau}(x)$ — а это **не** $\mathrm{Bel}(E)$. Правильная конструкция: Bel — это правое расширение Кана вдоль **профунктора дополнения** $J^{\complement}(E, x) = [x \notin E]$:

$$\mathrm{Ran}_{J^{\complement}}\,\bar{\tau}\,(E) = \int_{x \in \mathbf{X}} [0,1]\bigl([x \notin E],\, \bar{\tau}(x)\bigr) = \inf_{x \notin E} \bar{\tau}(x) = \mathrm{Bel}(E)\;\square$$

(внутренний hom: $[0,1](1, t) = t$, $[0,1](0, t) = 1$, поэтому end пробегает ровно $x \notin E$).

**Почему именно дополнение.** Дуальная согласованность Пытьева $\mathrm{Bel}(E) = \theta(\mathrm{Pl}(X \setminus E))$ — это утверждение, что инволюция $\theta(t) = 1 - t$ есть **дуальный изоморфизм кванталей** $([0,1], \max, \min) \xrightarrow{\;\sim\;} ([0,1], \min, \max)^{op}$, переводящий sup в inf и $\tau \mapsto \bar{\tau}$. Профунктор дополнения — это в точности образ $J$ под действием $\theta$ на истинностных значениях: $[x \notin E] = \theta_{\{0,1\}}([x \in E])$. Итого пара (Pl, Bel) — это пара (Lan вдоль $J$, Ran вдоль $\theta J$), и «гипотеза единства» сводится к коммутированию $\theta$ с парой расширений Кана — что проверяется на конечных примерах ниже.

**Уровень 3 — числовой пример:**

$X = \{a,b,c\}$, $\tau: a \mapsto 1.0,\, b \mapsto 0.6,\, c \mapsto 0.3$, $E = \{a,b\}$:

| Метод | Формула | Pl(E) | Bel(E) |
|-------|---------|-------|--------|
| Пытьев прямо | $\sup_{x\in E}\tau(x)$, $\inf_{x\notin E}\bar\tau(x)$ | $1.0$ | $0.7$ |
| Топология Скотта | $\mathrm{Int}_{\mathcal{T}_{\uparrow}}(E)$, $\mathrm{Cl}_{\mathcal{T}_{\downarrow}}(E)$ | $1.0$ | $0.7$ |
| Расширения Кана | $\mathrm{Lan}_J\tau(E)$, $\mathrm{Ran}_J\bar\tau(E)$ | $1.0$ | $0.7$ |

## Центральная гипотеза

> **Гипотеза** (к проверке). Пусть $\mathcal{V} = ([0,1], \min, 1)$. Тогда существует
> $\mathcal{V}$-обогащённый функтор $\Phi: \mathbf{Frm}_{[0,1]} \to \mathbf{BiTop}$
> такой, что квадрат
> $$\tau \xrightarrow{\mathrm{Lan}_J} \mathrm{Pl}\; =\; \mathrm{Int}_{\mathcal{T}_{\uparrow}} \xleftarrow{\Phi} (X, \mathcal{T}_{\uparrow}, \mathcal{T}_{\downarrow})$$
> коммутирует, и пара $(\mathrm{Pl}, \mathrm{Bel})$ является образом Isbell adjunction
> $\mathcal{O} \dashv \mathrm{Spec}$ применённого к $\tau$ над $\mathcal{V}$.

## Таблица статусов: что доказано, что открыто

| Утверждение | Статус | Источник |
|-------------|--------|----------|
| $[0,1]$ с $\min$ — commutative quantale (фрейм) | ✅ Доказано | nLab: quantale |
| $[0,1]$ несёт топологии Скотта $\mathcal{T}_{\uparrow}$, $\mathcal{T}_{\downarrow}$ | ✅ Доказано | — |
| $\tau^{-1}(\mathcal{T}_{\uparrow})$ и $\tau^{-1}(\mathcal{T}_{\downarrow})$ — битопос на $X$ | ✅ Доказано | — |
| $\mathrm{Lan}_J\,\tau = \mathrm{Pl}$ на дискретном $X$ | ✅ Доказано | coend = sup |
| $\mathrm{Ran}_J\,\bar{\tau} = \mathrm{Bel}$ на дискретном $X$ | ✅ Доказано | end = inf |
| $\mathrm{Pl}(E) = \mathrm{Int}_{\mathcal{T}_{\uparrow}}(E)$ на дискретном $X$ | ✅ Доказано | теорема выше |
| $\mathrm{Bel}(E) = \mathrm{Cl}_{\mathcal{T}_{\downarrow}}(E)$ на дискретном $X$ | ✅ Доказано | теорема выше |
| Isbell duality существует для любых $\mathcal{V}$-категорий | ✅ | nLab: Isbell duality |
| BiHeyting из $\mathcal{V}$-enrichment, а не только из топоса | ✅ через $a \multimap b$ | quantale hom |
| $\Phi: \mathbf{Frm}_{[0,1]} \to \mathbf{BiTop}$ — полный функтор | ⚓ Гипотеза | — |
| Isbell monad над $[0,1]$ = пара $(\mathrm{Pl}, \mathrm{Bel})$ | ⚓ Гипотеза | — |
| Обобщение на непрерывный $X$ (не дискретный) | ⚓ Открыто | — |

## Почему это важно: место в большой картине

Двойственность Исбелла — это **общий шаблон** дуальностей «геометрия ↔ алгебра» в математике.
Если гипотеза верна, теория Пытьева вписывается в один ряд:

| Дуальность | Геометрическая сторона | Алгебраическая сторона |
|------------|----------------------|------------------------|
| Stone duality | топологические пространства | булевы алгебры |
| Gelfand duality | компактные хаусдорфовы пространства | $C^*$-алгебры |
| Locale duality | локали | фреймы |
| **Пытьев (гипотеза)** | **битопос $(X, \mathcal{T}_{\uparrow}, \mathcal{T}_{\downarrow})$** | **пресноп $\tau: X \to [0,1]$** |

Общий объект, унифицирующий все строчки: **Isbell adjunction над соответствующим $\mathcal{V}$**.

**Границы.** Это исследовательская гипотеза, а не теорема из Пытьева; статус утверждений различается (часть доказана в примерах раздела 15, часть — программа).

> **Статус гипотезы:** в части пары $(mathrm{Pl}, mathrm{Bel})$ — доказана (изоморфизм категорий, включая бесконечный и бесточечный случаи): см. [PytevIso.ipynb](PytevIso.ipynb). Открытой остаётся функториальная часть ($Phi$ на морфизмах в $mathbf{BiTop}$) за пределами разобранных этажей.

In [9]:
-- | Раздел 14+: Bel = Ran вдоль профунктора дополнения (библиотека: lanAlong/ranAlong)

demoBelKan :: IO ()
demoBelKan = do
  putStrLn "=== Razdel 14+: Bel cherez Ran vdol dopolneniya ==="
  let dom = "abc"
      tau c = case c of { 'a' -> 1.0; 'b' -> 0.6; _ -> 0.2 }
      tauBar = (1 -) . tau
      subs = filterM (const [True, False]) dom
      pl  e = unUI (plMeasure dom (ui . tau) e)
      bel e = unUI (belMeasure dom (ui . tauBar) e)
      chkDual = all (\e -> ui (bel e) =~ theta (ui (pl (filter (`notElem` e) dom)))) subs
  putStrLn $ "  Bel(E) = theta(Pl(X\\E)) na vseh 8 podmnozhestvah: " ++ show chkDual
  putStrLn $ "  theta - dualnyj izomorfizm (max<->min): " ++ show (isDualIso theta)
  mapM_ (\e -> putStrLn $ "    E=" ++ show e ++ "  Pl=" ++ show (pl e)
               ++ "  Bel=" ++ show (bel e)) subs

demoBelKan

=== Razdel 14+: Bel cherez Ran vdol dopolneniya ===
  Bel(E) = theta(Pl(X\E)) na vseh 8 podmnozhestvah: True
  theta - dualnyj izomorfizm (max<->min): True
    E="abc"  Pl=1.0  Bel=1.0
    E="ab"  Pl=1.0  Bel=0.8
    E="ac"  Pl=1.0  Bel=0.4
    E="a"  Pl=1.0  Bel=0.4
    E="bc"  Pl=0.6  Bel=0.0
    E="b"  Pl=0.6  Bel=0.0
    E="c"  Pl=0.2  Bel=0.0
    E=""  Pl=0.0  Bel=0.0

In [10]:
-- | Разделы 13-14: квантале [0,1], топологии Скотта, Lan/Ran — из библиотеки

demoS1314 :: IO ()
demoS1314 = do
  putStrLn "=== Razdely 13-14: quantale, Scott, Kan ==="
  -- законы квантале: одна полиморфная проверка для [0,1] и Bool
  let gridS = [ui 0, ui 0.25, ui 0.5, ui 0.75, ui 1]
  putStrLn $ "  Residuation adj [0,1]: " ++ show (checkResiduationAdj gridS)
  putStrLn $ "  Residuation adj Bool:  " ++ show (checkResiduationAdj [False, True])
  putStrLn $ "  Frame distributivity:  " ++ show (checkFrameDistributivity gridS)
  -- топологии Скотта, индуцированные tau
  let tau c = case c of { 'a' -> 1.0; 'b' -> 0.6; _ -> 0.2 }
  putStrLn $ "  T_up   (t=0.5): " ++ show (scottUpOnX tau 0.5 "abc")
  putStrLn $ "  T_down (t=0.5): " ++ show (scottDownOnX tau 0.5 "abc")
  -- Lan вдоль принадлежности = Pl; Ran вдоль дополнения = Bel
  let dom = "abc"
  putStrLn $ "  Lan{a,b} = Pl  = " ++ show (unUI (plMeasure dom (ui . tau) "ab"))
  putStrLn $ "  Ran{a,b} = Bel = " ++ show (unUI (belMeasure dom (ui . (1 -) . tau) "ab"))

demoS1314

=== Razdely 13-14: quantale, Scott, Kan ===
  Residuation adj [0,1]: True
  Residuation adj Bool:  True
  Frame distributivity:  True
  T_up   (t=0.5): "ab"
  T_down (t=0.5): "c"
  Lan{a,b} = Pl  = 1.0
  Ran{a,b} = Bel = 0.8

---

<a id="subj-pytev-iso"></a>
## 14.5 Изоморфность Категорных Представлений → [PytevIso.ipynb](PytevIso.ipynb)

Доказательства состоятельности и эквивалентности категорных представлений теории Пытьева вынесены в отдельный ноутбук-сноску **[PytevIso.ipynb](PytevIso.ipynb)**: теоремы T1–T4 (универсальные свойства, единственность, θ-сопряжение); **изоморфизм категорий** битопос ≅ Кан ($Lambda circ K = B$, натуральность = п. 1.4); бесконечный случай (полная макситивность = плотность; ample fields Де Кумана = CABA); бесточечная теорема «мера = сопряжение Галуа»; d-меры на d-frames Юнга–Мошье ($mathrm{Bel} le mathrm{Pl}$ выводится из tot; FOUR Белнапа; Bel без дополнения; внешняя регуляризация Верваата). Гипотеза единства из §14 в части пары $(mathrm{Pl}, mathrm{Bel})$ — там доказана; открытые тропы (σ-зазор, идемпотентный анализ, монадная) перечислены в §3 сноски.

# 15. Три Сравнительных Примера: Битопос vs Расширения Кана

> Цель: показать, как два подхода — битопологический и через расширения Кана — описывают одни и те же объекты, где совпадают и где расходятся.

| Пример | Сложность | Что показывает |
|--------|-----------|----------------|
| 1 | Простой | Дискретный X, монотонная $\tau$. Оба совпадают — точка отсчёта |
| 2 | Средний | Отображение $\varphi: X \to Y$. Кан даёт формулу раздела 3 автоматически |
| 3 | Сложный | X — poset, $[0,1]$-обогащённая категория. Подходы расходятся |

## 15.1 Пример 1 (Простой): Дискретный X

$X = \{\text{low}, \text{mid}, \text{high}\}$, $\tau$ монотонно возрастает: $0.3 < 0.7 < 1.0$.

### 🔺 Битопологический взгляд

$\mathcal{T}_{\uparrow}^X$ — прообразы открытых $(t,1]$:

| t | Открытое мн-во |
|-----|----------------|
| 0.7 | {high} |
| 0.3 | {mid, high} |
| 0 | X |

$\mathrm{Pl}(E) = \mathrm{Int}_{\mathcal{T}_{\uparrow}}(E) = \sup_{x \in E} \tau(x)$

### 🔺 Расширение Кана

$J: X \hookrightarrow \mathcal{P}(X)$, $J(x) = \{x\}$. Coend = $\sup$:

$$\mathrm{Lan}_J\,\tau\,(E) = \int^{x} [x \in E] \otimes_{\min} \tau(x) = \sup_{x \in E} \tau(x)$$

![Пример 1: дискретный X](../diagrams/subj/subj_example1.svg)

In [11]:
-- Пример 1: дискретный X = {Low, Mid, High}

data Level = Low | Mid | High deriving (Show, Eq, Ord, Enum, Bounded)

tauEx1 :: Level -> Double
tauEx1 Low  = 0.3
tauEx1 Mid  = 0.7
tauEx1 High = 1.0

tauBarEx1 :: Level -> Double
tauBarEx1 x = 1.0 - tauEx1 x  -- простейшая дуальная согласованность: theta = (1-)

-- Битопологический подход
-- T-up: открытые = {x : tau(x) > t}
scottOpenUp :: Double -> [Level] -> [Level]
scottOpenUp t xs = filter (\x -> tauEx1 x > t) xs

-- Pl = sup tau на E
plBitopo :: [Level] -> Double
plBitopo []  = 0.0
plBitopo xs  = maximum (map tauEx1 xs)

-- Bel = inf tauBar на X\E
belBitopo :: [Level] -> Double
belBitopo xs =
  let compl = filter (\x -> x `notElem` xs) [minBound..maxBound]
  in if null compl then 1.0 else minimum (map tauBarEx1 compl)

-- Расширение Кана: Lan_J tau (E) = sup_{x in E} min(1, tau(x))
lanKan :: [Level] -> Double
lanKan []  = 0.0
lanKan xs  = maximum [min 1.0 (tauEx1 x) | x <- xs]

-- Ran_J tauBar (E) = inf_{x not in E} max(hom([0,1])(1, tauBar x), ...)
-- На дискретном X: Ran = inf_{x not in E} tauBar(x) = Bel(E)
ranKan :: [Level] -> Double
ranKan xs =
  let compl = filter (\x -> x `notElem` xs) [minBound..maxBound]
  in if null compl then 1.0 else minimum (map tauBarEx1 compl)

demo_ex1 :: IO ()
demo_ex1 = do
  putStrLn "=== Пример 1: дискретный X ==="
  putStrLn ""
  let allE = [[Low,Mid], [Mid,High], [Low], [High], [Low,Mid,High]]
  putStrLn "E                    | Pl (битопос) | Pl (Кан) | Bel (битопос) | Bel (Кан)"
  putStrLn (replicate 80 '-')
  mapM_ (\e -> do
    let pb = plBitopo e
        pk = lanKan e
        bb = belBitopo e
        bk = ranKan e
        match = if abs (pb-pk) < 1e-9 && abs (bb-bk) < 1e-9 then "== СОВПАДАЮТ ==" else "!! РАСХОДЯТСЯ !!"
    putStrLn (show e ++ replicate (20 - length (show e)) ' '
              ++ "| " ++ show pb ++ "        | " ++ show pk
              ++ "   | " ++ show bb ++ "          | " ++ show bk
              ++ "  " ++ match)) allE
  putStrLn ""
  putStrLn "T-up открытые при t=0.5:"
  print (scottOpenUp 0.5 [minBound..maxBound])

demo_ex1

Line 16: Eta reduce
Found:
scottOpenUp t xs = filter (\ x -> tauEx1 x > t) xs
Why not:
scottOpenUp t = filter (\ x -> tauEx1 x > t)Line 26: Avoid lambda using `infix`
Found:
(\ x -> x `notElem` xs)
Why not:
(`notElem` xs)Line 38: Avoid lambda using `infix`
Found:
(\ x -> x `notElem` xs)
Why not:
(`notElem` xs)

=== Пример 1: дискретный X ===

E                    | Pl (битопос) | Pl (Кан) | Bel (битопос) | Bel (Кан)
--------------------------------------------------------------------------------
[Low,Mid]           | 0.7        | 0.7   | 0.0          | 0.0  == СОВПАДАЮТ ==
[Mid,High]          | 1.0        | 1.0   | 0.7          | 0.7  == СОВПАДАЮТ ==
[Low]               | 0.3        | 0.3   | 0.0          | 0.0  == СОВПАДАЮТ ==
[High]              | 1.0        | 1.0   | 0.30000000000000004          | 0.30000000000000004  == СОВПАДАЮТ ==
[Low,Mid,High]      | 1.0        | 1.0   | 1.0          | 1.0  == СОВПАДАЮТ ==

T-up открытые при t=0.5:
[Mid,High]

![Diagram: Example 1 Discrete X](../diagrams/subjective/subj_example1.svg)

---

## 15.2 Пример 2 (Средний): Отображение $\varphi: X \to Y$

$X = \{\text{TempLow, TempMed, VibLow, VibHigh}\}$ с $\tau$,  
$Y = \{\text{Normal, Fault}\}$, $\varphi$ — диагностическое правило.

### 🔺 Битопологический взгляд

Нужно вычислить прямой образ топологии $\varphi_*(\mathcal{T}_{\uparrow}^X)$ и пересчитать $\mathrm{Pl}_Y$.

### 🔺 Расширение Кана

Формула из **раздела 3** ноутбука возникает автоматически как левое расширение Кана вдоль $\varphi$:

$$\tau^{\tilde{y}}(y) = \mathrm{Lan}_\varphi\,\tau\,(y) = \sup_{\varphi(x)=y} \tau(x)$$

![Пример 2: отображение phi](../diagrams/subj/subj_example2.svg)

In [12]:
-- Пример 2: phi: X -> Y, проталкивание tau

data Sensor  = TempLow | TempMed | VibLow | VibHigh deriving (Show, Eq, Ord, Enum, Bounded)
data Status  = Normal  | Fault              deriving (Show, Eq, Ord, Enum, Bounded)

tauX :: Sensor -> Double
tauX TempLow  = 0.9
tauX TempMed  = 0.5
tauX VibLow   = 0.7
tauX VibHigh  = 0.3

-- phi: диагностическое правило
phi :: Sensor -> Status
phi TempLow  = Normal
phi TempMed  = Normal
phi VibLow   = Normal
phi VibHigh  = Fault

-- Lan_phi tau: формула раздела 3 Пытьева
lanPhi :: Status -> Double
lanPhi y = maximum [tauX x | x <- [minBound..maxBound], phi x == y]

-- Битопологический подход: tau_Y(y) = sup_{phi(x)=y} tau(x)
-- Для дискретного X: прямой образ топологии совпадает с той же формулой
plYBitopo :: Status -> Double
plYBitopo y = maximum [tauX x | x <- [minBound..maxBound], phi x == y]

demo_ex2 :: IO ()
demo_ex2 = do
  putStrLn "=== Пример 2: phi: X -> Y ==="
  putStrLn ""
  putStrLn "tau на X:"
  mapM_ (\x -> putStrLn ("  " ++ show x ++ " -> " ++ show (tauX x) ++ "  phi -> " ++ show (phi x)))
        [minBound..maxBound :: Sensor]
  putStrLn ""
  putStrLn "Проталкивание tau вдоль phi:"
  mapM_ (\y -> do
    let pk = lanPhi y
        pb = plYBitopo y
        match = if abs (pk-pb) < 1e-9 then "== совпадают ==" else "!! РАСХОДЯТСЯ !!"
    putStrLn ("  " ++ show y ++ ": Lan_phi=" ++ show pk ++ "  Bitopo=" ++ show pb ++ "  " ++ match))
    [minBound..maxBound :: Status]
  putStrLn ""
  putStrLn "Вывод: tau_Y(y) = sup_{phi(x)=y} tau(x) (раздел 3 Пытьева)"
  putStrLn "       = Lan_phi tau.  Автоматически."

demo_ex2

=== Пример 2: phi: X -> Y ===

tau на X:
  TempLow -> 0.9  phi -> Normal
  TempMed -> 0.5  phi -> Normal
  VibLow -> 0.7  phi -> Normal
  VibHigh -> 0.3  phi -> Fault

Проталкивание tau вдоль phi:
  Normal: Lan_phi=0.9  Bitopo=0.9  == совпадают ==
  Fault: Lan_phi=0.3  Bitopo=0.3  == совпадают ==

Вывод: tau_Y(y) = sup_{phi(x)=y} tau(x) (раздел 3 Пытьева)
       = Lan_phi tau.  Автоматически.

![Diagram: Example 2 phi](../diagrams/subjective/subj_example2.svg)

---

## 15.3 Пример 3 (Сложный): X — Poset, $[0,1]$-обогащённая Категория

X — поset состояний: $\text{OK} \leq \text{Warn} \leq \text{Fail}$ и $\text{OK} \leq \text{Degrade} \leq \text{Fail}$.  
$\tau$ монотонна по порядку: $\tau(\text{OK})=0.2,\; \tau(\text{Warn})=0.6,\; \tau(\text{Degrade})=0.8,\; \tau(\text{Fail})=1.0$.

**Ключевое отличие:** $X$ теперь не дискретная $[0,1]$-категория. Хом-объекты:
$$\mathbf{X}(x, y) = [x \leq y] \in \{0, 1\} \subset [0,1]$$

Обогащённое левое расширение Кана:
$$\mathrm{Lan}_J^{\mathcal{V}}\,\tau\,(E) = \sup_{x \in X} \min\bigl(\sup_{e \in E} \mathbf{X}(x, e),\; \tau(x)\bigr)$$

Когда $\mathbf{X}(x,e) \in \{0,1\}$ — совпадает с дискретным. Но если перейти к непрерывному порядку (например, $\mathbf{X}(x,y) = \tau(y) - \tau(x)$ при $x \leq y$) — расходится.

### 🔺 Где расходятся?

Битопос строится по $\tau$, не зная про $\mathbf{X}(x,y)$.  
Обогащённый Кан использует $\mathbf{X}(x,y)$ как «вес перехода».  
При $\mathbf{X}(x,y) = \tau(y)/\tau(x)$ (нормированный) — расхождение становится измеримым.

![Пример 3: poset X](../diagrams/subj/subj_example3.svg)

In [13]:
-- Пример 3: X как poset, [0,1]-обогащённая категория

data State3 = OK3 | Warn3 | Degrade3 | Fail3 deriving (Show, Eq, Ord, Enum, Bounded)

tauP :: State3 -> Double
tauP OK3      = 0.2
tauP Warn3    = 0.6
tauP Degrade3 = 0.8
tauP Fail3    = 1.0

-- Порядок: OK <= Warn <= Fail, OK <= Degrade <= Fail
leq :: State3 -> State3 -> Bool
leq OK3      _         = True
leq Warn3    Warn3     = True
leq Warn3    Fail3     = True
leq Degrade3 Degrade3  = True
leq Degrade3 Fail3     = True
leq Fail3    Fail3     = True
leq _        _         = False

-- Хом-объекты трёх вариантов [0,1]-обогащения:

-- Вариант A: дискретная категория X(x,y) = [x==y]
homDiscrete :: State3 -> State3 -> Double
homDiscrete x y = if x == y then 1.0 else 0.0

-- Вариант B: poset X(x,y) = [x<=y] in {0,1}
homPoset :: State3 -> State3 -> Double
homPoset x y = if leq x y then 1.0 else 0.0

-- Вариант C: "непрерывный" вес перехода X(x,y) = tau(y)/tau(x) при x<=y
homContinuous :: State3 -> State3 -> Double
homContinuous x y
  | leq x y   = tauP y / tauP x
  | otherwise = 0.0

-- Обобщённый обогащённый Lan_J tau (E) через заданный hom
-- Lan_J tau (E) = sup_x min(sup_{e in E} hom(x,e), tau(x))
lanEnriched :: (State3 -> State3 -> Double) -> [State3] -> Double
lanEnriched hom e
  | null e    = 0.0
  | otherwise = maximum
      [ min (maximum [hom x ex | ex <- e]) (tauP x)
      | x <- [minBound..maxBound] ]

-- Pl обычный (битопос, без структуры X)
plFlat :: [State3] -> Double
plFlat [] = 0.0
plFlat xs = maximum (map tauP xs)

demo_ex3 :: IO ()
demo_ex3 = do
  putStrLn "=== Пример 3: poset X, три варианта обогащения ==="
  putStrLn ""
  let testSets = [ [Warn3, Fail3]
                 , [OK3, Degrade3]
                 , [Warn3, Degrade3]
                 , [Fail3]
                 , [OK3] ]
  putStrLn "E                  | Pl(flat) | Lan(дискр) | Lan(poset) | Lan(непрерыв)"
  putStrLn (replicate 80 '-')
  mapM_ (\e -> do
    let pf = plFlat e
        ld = lanEnriched homDiscrete e
        lp = lanEnriched homPoset e
        lc = lanEnriched homContinuous e
    putStrLn (show e ++ replicate (18 - length (show e)) ' '
              ++ "| " ++ show pf
              ++ "   | " ++ show ld
              ++ "      | " ++ show lp
              ++ "      | " ++ show lc)) testSets
  putStrLn ""
  putStrLn "Наблюдения:"
  putStrLn "  Lan(дискр) == Pl(flat)        -- дискретный X = без структуры"
  putStrLn "  Lan(poset) == Pl(flat)        -- poset {0,1}: coend = sup, тот же результат"
  putStrLn "  Lan(непрерыв) может отличаться -- вес перехода масштабирует tau(x)"
  putStrLn ""
  putStrLn "Пример: E = [Warn3]"
  let e = [Warn3]
  putStrLn ("  Pl(flat)  = " ++ show (plFlat e))
  putStrLn ("  Lan(poset) = " ++ show (lanEnriched homPoset e))
  putStrLn ("  Lan(непрерыв) = " ++ show (lanEnriched homContinuous e))
  putStrLn "  (OK->Warn: hom = 0.6/0.2 = 3.0, но min(..., tau(OK)=0.2) = 0.2; Warn->Warn: 1.0, tau=0.6)"

demo_ex3

=== Пример 3: poset X, три варианта обогащения ===

E                  | Pl(flat) | Lan(дискр) | Lan(poset) | Lan(непрерыв)
--------------------------------------------------------------------------------
[Warn3,Fail3]     | 1.0   | 1.0      | 1.0      | 1.0
[OK3,Degrade3]    | 0.8   | 0.8      | 0.8      | 0.8
[Warn3,Degrade3]  | 0.8   | 0.8      | 0.8      | 0.8
[Fail3]           | 1.0   | 1.0      | 1.0      | 1.0
[OK3]             | 0.2   | 0.2      | 0.2      | 0.2

Наблюдения:
  Lan(дискр) == Pl(flat)        -- дискретный X = без структуры
  Lan(poset) == Pl(flat)        -- poset {0,1}: coend = sup, тот же результат
  Lan(непрерыв) может отличаться -- вес перехода масштабирует tau(x)

Пример: E = [Warn3]
  Pl(flat)  = 0.6
  Lan(poset) = 0.6
  Lan(непрерыв) = 0.6
  (OK->Warn: hom = 0.6/0.2 = 3.0, но min(..., tau(OK)=0.2) = 0.2; Warn->Warn: 1.0, tau=0.6)

![Diagram: Example 3 Poset](../diagrams/subjective/subj_example3.svg)

---

## 15.4 Сравнительный итог

| | Пример 1 (дискрет.) | Пример 2 (φ) | Пример 3 (poset) |
|-|---------------------|--------------|------------------|
| **Битопос** | ✅ даёт Pl, Bel | требует `φ_*(T)` | видит только `τ`, не `X(x,y)` |
| **Кан (дискр.)** | ✅ совпадает | `Lan_φ τ` — автоматически | = Pl(flat) |
| **Кан (обогащ.)** | = дискрет. | = дискрет. (если J фиксирован) | зависит от `X(x,y)` |

**Вывод:**
- На дискретном $X$ все подходы эквивалентны.
- Обогащённый Кан естественно обобщает формулу раздела 3 на произвольные $\varphi: X \to Y$.
- При непрерывном поряд. весе `X(x,y)` расширение Кана учитывает структуру $X$, недоступную битопосу.
- Это указывает на направление обобщения: **заменить $\mathbf{X}(x,y) \in \{0,1\}$ на $[0,1]$-значный вес** и изучить соответствующую субъективную модель.

In [14]:
-- | Совместимость: прежний внутриноутбучный API для раздела 16.
-- Библиотечные plMeasure/belMeasure (lib/KanExtension.hs) принимают домен,
-- распределение и событие; здесь восстанавливаем старую двухаргументную
-- форму (событие + распределение), перекрывая импорт — GHCi это разрешает.

plMeasure :: [a] -> (a -> Double) -> Double
plMeasure xs tau = maximum (0 : map tau xs)

belMeasure :: [a] -> (a -> Double) -> Double
belMeasure xsComp tauBar = minimum (1 : map tauBar xsComp)

putStrLn "compat OK: plMeasure/belMeasure (legacy signature)"

compat OK: plMeasure/belMeasure (legacy signature)

# 16. Практический пример: диагностика двигателя (3 эксперта)

Сведём аппарат субъективной модели на содержательной задаче. Три эксперта оценивают состояние
двигателя из множества $\{$OK, фильтр, подшипник, критическое$\}$, задавая распределения
правдоподобия $\tau$ и доверия $\bar\tau$. Мы комбинируем оценки с весами, считаем меры
события, интегралы риска (severity), применяем принцип относительности ($\gamma=\sqrt{\cdot}$)
и строим условную модель после показаний датчика.

Пример использует API субъективной модели из разделов 1--5 (`SubjModel`, `plMeasure`,
`belMeasure`, `plIntegral`, `belIntegral`).

In [15]:
-- S16: Практический пример — диагностика двигателя (3 эксперта)
-- Используем тип SubjModel и операции из разделов 1-5.

data Engine = EOK | EFilter | EBearing | ECrit deriving (Show, Eq, Ord, Enum, Bounded)

engStates :: [Engine]
engStates = [minBound .. maxBound]

-- Построить модель эксперта из таблиц; нормируем sup tau = 1
mkExpert :: [(Engine, Double)] -> [(Engine, Double)] -> SubjModel Engine
mkExpert plT belT = SubjModel engStates tau tbar
  where
    mx = maximum (map snd plT)
    tau x  = maybe 0 (/ mx) (lookup x plT)
    tbar x = maybe 0 id (lookup x belT)

expertA, expertB, expertC :: SubjModel Engine
expertA = mkExpert [(EOK,0.3),(EFilter,0.9),(EBearing,0.5),(ECrit,0.1)]
                   [(EOK,0.1),(EFilter,0.0),(EBearing,0.2),(ECrit,0.6)]
expertB = mkExpert [(EOK,0.2),(EFilter,0.7),(EBearing,0.8),(ECrit,0.3)]
                   [(EOK,0.2),(EFilter,0.1),(EBearing,0.0),(ECrit,0.5)]
expertC = mkExpert [(EOK,0.1),(EFilter,0.6),(EBearing,0.9),(ECrit,0.4)]
                   [(EOK,0.3),(EFilter,0.2),(EBearing,0.0),(ECrit,0.4)]

-- Комбинирование экспертов: взвешенная сумма tau/tauBar, затем нормировка sup tau = 1
combineExperts :: [(SubjModel Engine, Double)] -> SubjModel Engine
combineExperts ws = SubjModel engStates tau tbar
  where
    wsum      = sum (map snd ws)
    rawTau x  = sum [w * smTau m x    | (m, w) <- ws] / wsum
    rawTBar x = sum [w * smTauBar m x | (m, w) <- ws] / wsum
    mx        = maximum (map rawTau engStates)
    tau x     = if mx == 0 then rawTau x else rawTau x / mx
    tbar      = rawTBar

combined :: SubjModel Engine
combined = combineExperts [(expertA, 0.4), (expertB, 0.35), (expertC, 0.25)]

-- Условная модель: ограничить домен наблюдаемым событием и перенормировать
conditionModel :: SubjModel Engine -> [Engine] -> SubjModel Engine
conditionModel m obs = SubjModel obs tau (smTauBar m)
  where
    mx    = maximum (0 : map (smTau m) obs)
    tau x = if mx == 0 then smTau m x else smTau m x / mx

-- Серьёзность состояния (функция потерь для интеграла Сугено)
severity :: Engine -> Double
severity EOK      = 0.0
severity EFilter  = 0.3
severity EBearing = 0.6
severity ECrit    = 1.0

demo_engine :: IO ()
demo_engine = do
  putStrLn "=== Диагностика двигателя: 3 эксперта ==="
  let warn     = [EFilter, EBearing]
      compWarn = filter (`notElem` warn) engStates
  putStrLn $ "Pl({Filter,Bearing})  = " ++ show (plMeasure warn (smTau combined))
  putStrLn $ "Bel({Filter,Bearing}) = " ++ show (belMeasure compWarn (smTauBar combined))
  putStrLn $ "Зазор Pl - Bel        = "
           ++ show (plMeasure warn (smTau combined) - belMeasure compWarn (smTauBar combined))
  putStrLn ""
  putStrLn "-- Интегралы риска (severity) --"
  putStrLn $ "Pl-интеграл (оптимист):   " ++ show (plIntegral engStates (smTau combined) severity)
  putStrLn $ "Bel-интеграл (пессимист): " ++ show (belIntegral engStates (smTauBar combined) severity)
  putStrLn ""
  putStrLn "-- Принцип относительности (gamma = sqrt сохраняет порядок) --"
  let g = combined { smTau = sqrt . smTau combined }  -- принцип относительности
  putStrLn $ "Pl({Filter}) до:    " ++ show (plMeasure [EFilter] (smTau combined))
  putStrLn $ "Pl({Filter}) после: " ++ show (plMeasure [EFilter] (smTau g))
  putStrLn ""
  putStrLn "-- Условная модель (датчик исключил Critical) --"
  let c = conditionModel combined [EOK, EFilter, EBearing]
  putStrLn $ "Pl(Bearing | not Crit) = " ++ show (plMeasure [EBearing] (smTau c))
  putStrLn $ "Pl(Filter  | not Crit) = " ++ show (plMeasure [EFilter]  (smTau c))

demo_engine

Line 15: Use fromMaybe
Found:
maybe 0 id
Why not:
Data.Maybe.fromMaybe 0Line 53: Use camelCase
Found:
demo_engine :: IO ()
Why not:
demoEngine :: IO ()Line 54: Use camelCase
Found:
demo_engine = ...
Why not:
demoEngine = ...

=== Диагностика двигателя: 3 эксперта ===
Pl({Filter,Bearing})  = 1.0
Bel({Filter,Bearing}) = 0.185
Зазор Pl - Bel        = 0.815

-- Интегралы риска (severity) --
Pl-интеграл (оптимист):   0.6
Bel-интеграл (пессимист): 0.185

-- Принцип относительности (gamma = sqrt сохраняет порядок) --
Pl({Filter}) до:    1.0
Pl({Filter}) после: 1.0

-- Условная модель (датчик исключил Critical) --
Pl(Bearing | not Crit) = 0.9419252187748607
Pl(Filter  | not Crit) = 1.0

# 17. Монада Возможности — Поссибилистский Двойник Монады Гири

В [Uncertainty.ipynb](Uncertainty.ipynb) вероятность оформлена как монада: дискретные распределения (раздел 2), монада Гири (раздел 3), марковские цепи в категории Клейсли (раздел 7). Теория Пытьева даёт **точный поссибилистский аналог** этой конструкции.

## Определение

$$T(X) = \{\tau: X \to [0,1] \mid \sup_x \tau(x) = 1\} \qquad\text{— распределения правдоподобий}$$

- **Единица** (дельта Дирака): $\eta(x) = \tau$, где $\tau(x) = 1$, $\tau(y) = 0$ при $y \neq x$ — «точное знание» Пытьева
- **bind**: для $k: X \to T(Y)$

$$(\tau \mathbin{>\!\!>\!\!=} k)(y) = \sup_{x \in X} \min\{\tau(x),\, k(x)(y)\}$$

Это **pl-интеграл** из раздела 5: bind — это в точности $\mathrm{pl}_{\tau}$, применённый к ядру $k$. Теорема 1.1 Пытьева (представимость pl-интеграла) — это утверждение о том, что $T$ — монада, индуцированная кванталью $([0,1], \min, 1)$.

## Сравнение с монадой Гири

| | Гири (вероятность) | Возможность (Пытьев) |
|---|---|---|
| Объект | $\sum_x p(x) = 1$ | $\sup_x \tau(x) = 1$ |
| Полукольцо | $(\mathbb{R}_{\ge 0}, +, \cdot)$ | $([0,1], \max, \min)$ |
| bind | $\sum_x p(x) \cdot k(x)(y)$ | $\sup_x \min(\tau(x), k(x)(y))$ |
| Клейсли-композиция | умножение стох. матриц | sup-min «умножение» матриц |
| Категория Клейсли | стохастические отображения | $[0,1]$-обогащённые отношения |
| Независимое произведение | $p \otimes q$ | $\min$-произведение (Опр. 1.2 Пытьева) |

Обе — частные случаи **монады распределений над коммутативным полукольцом**; замена $(+,\cdot) \to (\max,\min)$ переводит марковские цепи Uncertainty.ipynb в поссибилистские переходные системы. Как и `Dist` через `Map`, это *ограниченная* монада (нужен `Ord` для ключей) — инстанс класса `Monad` Haskell не даётся, но законы монады выполняются и проверяются ниже.

In [16]:
-- | Раздел 17: монада возможности — из библиотеки (../lib/Distribution.hs)

data W = Sun | Rain | Fog deriving (Show, Eq, Ord)

stepW :: W -> Poss W
stepW Sun  = possOf [(Sun, 1.0), (Fog, 0.4), (Rain, 0.2)]
stepW Rain = possOf [(Rain, 1.0), (Fog, 0.7), (Sun, 0.3)]
stepW Fog  = possOf [(Fog, 1.0), (Sun, 0.6), (Rain, 0.6)]

demoPossMonad :: IO ()
demoPossMonad = do
  putStrLn "=== Razdel 17: monada vozmozhnosti (i ne tolko) ==="
  let m0 = possOf [(Sun, 1.0), (Rain, 0.5)]
      k2 w = possOf [(w, 1.0), (Fog, 0.5)]
  putStrLn $ "  [Poss]  zakony monady: " ++ show (checkMonadLaws m0 stepW k2 Sun)
  putStrLn $ "  sup posle bind = " ++ show (supPoss (bindD m0 stepW))
  mapM_ (\n -> putStrLn $ "  " ++ show n ++ " shagov ot Sun: "
               ++ showDistList (nStepsD n stepW Sun)) [1, 2, 3]
  putStrLn $ "  Stabilizaciya sup-min stepenej: "
             ++ show (eqDist (nStepsD 3 stepW Sun) (nStepsD 4 stepW Sun))
  -- ТА ЖЕ монада над вероятностным полукольцом: одна математика, две теории
  let stepP :: W -> Dist ProbW W
      stepP Sun  = distOf [(Sun, ProbW 0.7), (Fog, ProbW 0.2), (Rain, ProbW 0.1)]
      stepP Rain = distOf [(Rain, ProbW 0.5), (Fog, ProbW 0.3), (Sun, ProbW 0.2)]
      stepP Fog  = distOf [(Fog, ProbW 0.4), (Sun, ProbW 0.3), (Rain, ProbW 0.3)]
      etaP w = distOf [(w, ProbW 1.0)]
  putStrLn $ "  [ProbW] zakony monady: "
             ++ show (checkMonadLaws (etaP Sun) stepP etaP Sun)
  putStrLn $ "  [ProbW] 2 shaga: " ++ showDistList (nStepsD 2 stepP Sun)
  -- И над Bool: семантика достижимости
  let stepB :: W -> Dist Bool W
      stepB Sun  = distOf [(Sun, True), (Fog, True)]
      stepB Rain = distOf [(Rain, True)]
      stepB Fog  = distOf [(Fog, True), (Rain, True)]
  putStrLn $ "  [Bool] dostizhimost za 2 shaga iz Sun: "
             ++ showDistList (nStepsD 2 stepB Sun)

demoPossMonad

=== Razdel 17: monada vozmozhnosti (i ne tolko) ===
  [Poss]  zakony monady: True
  sup posle bind = 1.0
  1 shagov ot Sun: [(Sun,UI 1.0),(Rain,UI 0.2),(Fog,UI 0.4)]
  2 shagov ot Sun: [(Sun,UI 1.0),(Rain,UI 0.4),(Fog,UI 0.4)]
  3 shagov ot Sun: [(Sun,UI 1.0),(Rain,UI 0.4),(Fog,UI 0.4)]
  Stabilizaciya sup-min stepenej: True
  [ProbW] zakony monady: True
  [ProbW] 2 shaga: [(Sun,ProbW 0.5699999999999998),(Rain,ProbW 0.18),(Fog,ProbW 0.25)]
  [Bool] dostizhimost za 2 shaga iz Sun: [(Sun,True),(Rain,True),(Fog,True)]

# 18. Обогащённая $\mathbf{X}$: Когда Двойственность Исбелла Перестаёт Быть Тривиальной

В разделе 14 категория $\mathbf{X}$ дискретна ($\mathbf{X}(x,y) = [x=y]$), поэтому Йонеда и Исбелл вырождаются: coend схлопывается в $\sup$, и вся конструкция лишь воспроизводит формулы Пытьева. Содержательная картина возникает, если МИ оценивает ещё и **степень неразличимости состояний**.

## $\mathbf{X}$ как обобщённое метрическое пространство Ловера

Зададим $[0,1]$-обогащённую категорию: $\mathbf{X}(x,y) \in [0,1]$ — «правдоподобие того, что $x$ и $y$ неразличимы», с аксиомами

$$\mathbf{X}(x,x) = 1, \qquad \min\{\mathbf{X}(x,y),\, \mathbf{X}(y,z)\} \le \mathbf{X}(x,z)$$

(рефлексивность и $\min$-транзитивность — это в точности категория, обогащённая над кванталью $([0,1],\min,1)$; как у Ловера, у которого вместо неё берётся $([0,\infty], +, 0)$ и получается обобщённая метрика).

## Преснопы = распределения, согласованные с неразличимостью

$[0,1]$-пресноп $\varphi: \mathbf{X}^{op} \to [0,1]$ обязан удовлетворять $\min\{\mathbf{X}(x,y), \varphi(y)\} \le \varphi(x)$: похожие состояния обязаны иметь похожие правдоподобия. Произвольное $\tau$ преснопом быть не обязано — его **йонедовское пополнение** (Lan вдоль Йонеды):

$$\hat{\tau}(x) = \sup_{y} \min\{\mathbf{X}(x,y),\, \tau(y)\}$$

— наименьший пресноп над $\tau$: субъективная модель «доразмывается» по неразличимости.

## Двойственность Исбелла: $\mathcal{O} \dashv \mathrm{Spec}$ в числах

$$\mathcal{O}(\varphi)(c) = \inf_{x} \bigl(\varphi(x) \multimap \mathbf{X}(x,c)\bigr), \qquad \mathrm{Spec}(\psi)(c) = \inf_{x} \bigl(\psi(x) \multimap \mathbf{X}(c,x)\bigr)$$

Единица сопряжения даёт $\varphi \le \mathrm{Spec}(\mathcal{O}(\varphi))$, и тройка $\mathcal{O}\,\mathrm{Spec}\,\mathcal{O} = \mathcal{O}$. **Неподвижные точки** ($\varphi = \mathrm{Spec}(\mathcal{O}(\varphi))$) — это Isbell envelope / tight span: кандидат на роль «объективного пополнения» субъективной модели — состояния, которые можно добавить к $X$, не нарушив согласованности правдоподобий с неразличимостью. Гипотеза раздела 14 в этой постановке: **пара (Pl, Bel) дуально согласована по Пытьеву $\iff$ ($\tau$, $\bar\tau$) — избелловски сопряжённая пара**. Ниже всё проверяется численно.

In [17]:
-- | Раздел 18: обогащённая X и Isbell O -| Spec — из библиотеки (../lib/KanExtension.hs)

data St = StA | StB | StC deriving (Show, Eq, Ord, Enum, Bounded)

homSt :: St -> St -> UnitInterval
homSt x y | x == y = ltop
homSt StA StB = ui 0.7
homSt StB StA = ui 0.7
homSt _   _   = ui 0.5

tauRawSt :: St -> UnitInterval
tauRawSt StA = ui 1.0
tauRawSt StB = ui 0.2   -- нарушение преснопа: hom(A,B)=0.7, tau(A)=1
tauRawSt StC = ui 0.1

demoEnriched :: IO ()
demoEnriched = do
  putStrLn "=== Razdel 18: obogashchyonnaya X i Isbell ==="
  let xs = [StA, StB, StC]
      showQ f = show [ (x, unUI (f x)) | x <- xs ]
  putStrLn $ "  Tranzitivnost hom:  " ++ show (isTransitive homSt xs)
  putStrLn $ "  tauRaw - presnop?   " ++ show (isPresheaf homSt xs tauRawSt)
  let tauHat = yonedaHat homSt xs tauRawSt
  putStrLn $ "  tauHat = " ++ showQ tauHat
  putStrLn $ "  tauHat - presnop?   " ++ show (isPresheaf homSt xs tauHat)
  putStrLn $ "  Edinica Isbell (phi <= Spec(O phi)): "
             ++ show (checkIsbellUnit homSt xs tauHat)
  putStrLn $ "  Treugolnik O Spec O = O: " ++ show (checkIsbellTriangle homSt xs tauHat)
  let fix1 = isbellSpec homSt xs (isbellO homSt xs tauHat)
  putStrLn $ "  Spec(O(tauHat)) = " ++ showQ fix1 ++ " (tight span)"

demoEnriched

=== Razdel 18: obogashchyonnaya X i Isbell ===
  Tranzitivnost hom:  True
  tauRaw - presnop?   False
  tauHat = [(StA,1.0),(StB,0.7),(StC,0.5)]
  tauHat - presnop?   True
  Edinica Isbell (phi <= Spec(O phi)): True
  Treugolnik O Spec O = O: True
  Spec(O(tauHat)) = [(StA,1.0),(StB,0.7),(StC,0.5)] (tight span)

# ✅ Итоги: Теория Субъективного Моделирования Пытьева

## Основные результаты

| Раздел | Ключевое понятие | Источник |
|--------|-----------------|----------|
| 1 | Пространство $(X, \mathcal{P}(X), \mathrm{Pl}^{\tilde{x}}, \mathrm{Bel}^{\tilde{x}})$ — субъективная модель НОЭ | Часть 1, п. 1.1 |
| 2 | Шкалы $L = ([0,1], \max, \min)$ и $\bar{L} = ([0,1], \min, \max)$ | Часть 1, п. 1.3 |
| 3 | Меры Pl и Bel через $\sup$ и $\inf$ распределений $\tau, \bar{\tau}$ | Часть 1, п. 1.1 |
| 4 | Принцип относительности — группа $\Gamma$ автоморфизмов | Часть 1, п. 1.3 |
| 5 | Pl- и bel-интегралы: $\sup\min$ и $\inf\max$ (Теорема 1.1) | Часть 1, п. 1.6 |
| 6 | Субъективная независимость $\equiv$ Pl-независимость | Часть 1, п. 1.7 |
| 7 | Условные распределения через субъективные шкалы | Часть 1, п. 1.8 |
| 8 | Три варианта мер: основной, с фикс. точками, психофизический | Часть 1, п. 1.9 |
| 9 | Эмпирическое восстановление НЧ.НОЭ по наблюдениям | Часть 1, разд. 2 |
| 10 | Комбинирование через матрицы парных сравнений | Часть 1, п. 2.2 |
| 11 | Энтропии: $H(\tau) = \mathrm{pl}(\bar{\tau})$, $H(\bar{\tau}) = \mathrm{bel}(\tau)$ | Часть 2, разд. 2 |
| 12 | Оптимальное правило идентификации НО.НЧ.О. | Часть 2, разд. 3 |
| 13 | $[0,1]$ — quantale/фрейм; топологии Скотта $\mathcal{T}_{\uparrow}, \mathcal{T}_{\downarrow}$ индуцируют битопос | Категорное |
| 14 | $\mathrm{Lan}_J\,\tau = \mathrm{Pl} = \mathrm{Int}_{\mathcal{T}_{\uparrow}}$; гипотеза: единство через Isbell duality над $[0,1]$ | Гипотеза |

## Ключевые формулы

$$\mathrm{Pl}(E) = \sup_{x \in E} \tau(x), \qquad \mathrm{Bel}(E) = \inf_{x \notin E} \bar{\tau}(x)$$

$$\mathrm{pl}_{\tau}(g) = \sup_x \min(\tau(x), g(x)), \qquad \mathrm{bel}_{\bar{\tau}}(\bar{g}) = \inf_x \max(\bar{\tau}(x), \bar{g}(x))$$

$$H(\tau) = \mathrm{pl}_{\tau}(\bar{\tau}), \qquad H(\bar{\tau}) = \mathrm{bel}_{\bar{\tau}}(\tau)$$

$$\mathrm{Lan}_J\,\tau\,(E) = \int^{x} J(x,E) \otimes_{\min} \tau(x) = \sup_{x \in E}\tau(x) = \mathrm{Pl}(E)$$

## Три вида знания

| Модель | $\tau(x)$ для всех $x$ | $\bar{\tau}(x)$ для всех $x$ |
|--------|----------------------|-----------------------------|
| Абсолютное незнание | $1$ | $0$ |
| Точное знание $x_0$ | $\mathbf{1}_{x=x_0}$ | $\mathbf{1}_{x=x_0}$ |
| Произвольная модель | $\tau: X \to [0,1]$ | $\bar{\tau}: X \to [0,1]$ |

## Статус категорной гипотезы

| Утверждение | Статус |
|-------------|--------|
| Coend = sup = Interior на дискретном $X$ | ✅ Доказано |
| End = inf = Closure на дискретном $X$ | ✅ Доказано |
| $[0,1]$ — quantale (фрейм) | ✅ Доказано |
| Isbell adjunction $\mathcal{O} \dashv \mathrm{Spec}$ над $[0,1]$ = пара (Pl, Bel) | ⚓ Гипотеза |
| Кондиционирование = residuation (правый сопряжённый к $\min(-,a)$) | ✅ Доказано + проверено |
| $\mathrm{Bel} = \mathrm{Ran}$ вдоль профунктора дополнения $\theta J$ | ✅ Дыра закрыта |
| Монада возможности (разд. 17): законы монады | ✅ Проверено численно |
| Isbell-сопряжение на обогащённой $\mathbf{X}$ (разд. 18) | ✅ Проверено численно |

В разделах 15–16 теория проверена делом: три сравнительных примера (битопос vs расширения Кана) и практическая диагностика двигателя тремя экспертами.

С ветки categorical-core-lib весь код ноутбука — вызовы библиотеки `lib/` (канонический источник: https://github.com/darklordshish/SubjectiveModeling, cabal-пакет с 33 тестами законов и двумя примерами).

---

**← Предыдущий:** [Неопределённость](Uncertainty.ipynb)
